# Préparation des données — Compagnon Immobilier

**Projet :** Compagnon Immobilier · **Date :** 12/10/2026, mis à jour le 31/10/2026

Ce notebook rassemble toute la préparation des données de l'équipe. Il se lance après `00_telechargement`, qui place les sources brutes dans `data/raw/`. Tout ce qu'il crée est enregistré dans `data/processed/`.

Il est organisé en blocs, dans l'ordre où ils doivent être exécutés :

1. **Table de liaison** : relier chaque code commune à son bassin de vie 2022 et à son sous-bassin ; superficie des communes.
2. **Filtrage de DVF+** : garder les ventes d'un seul logement ancien, dans un seul fichier.
3. **Échantillon d'apprentissage** : tirer un échantillon stratifié des ventes 2014-2024.
4. **Tables cibles** : prix médian au m² par commune, par bassin de vie et par sous-bassin, au mois, au trimestre et à l'année.
5. **Filosofi** : assembler les revenus des ménages par commune, de 2014 à 2021.
6. **Ménages** : structure des ménages par commune (recensement 2022).
7. **Parc de logement** : caractéristiques du parc par commune (recensement 2022).
8. **Délinquance** : faits enregistrés et taux par commune, de 2016 à 2025.
9. **LOVAC** : logements privés vacants par commune, millésimes 2020 à 2026.
10. **Équipements** : présence des équipements de la BPE par commune, millésimes 2015 et 2020.
11. **Taux des crédits immobiliers** : taux national par mois, trimestre et année, de 1998 à 2026.
12. **Équipements (BPE 2025)** : nombre d'équipements par commune et par domaine.

Les blocs 3 et 4 utilisent la population filtrée du bloc 2, et le bloc 4 utilise aussi la table de liaison du bloc 1. Les blocs 5 à 11 sont indépendants les uns des autres ; le bloc 12 utilise la table de liaison du bloc 1.

La version courte de ce code, sans explications, est dans `preparation.py`.

## Imports et chemins

Les chemins viennent du fichier `config.py`, à la racine du projet. Ils sont calculés à partir de l'emplacement du projet : ils sont donc les mêmes sur tous les ordinateurs.

Le notebook est rangé dans le dossier `01_preparation`. Python ne cherche pas `config.py` dans le dossier parent : on lui indique d'y regarder avec `sys.path.insert`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import re
# Le dossier parent du notebook est la racine du projet, où se trouve config.py
sys.path.insert(0, str(Path.cwd().parent))
from config import DATA_RAW, DATA_OUT

# Dossier de sortie, créé s'il n'existe pas encore
DATA_OUT.mkdir(parents=True, exist_ok=True)

print("Sources brutes :", DATA_RAW)
print("Sorties        :", DATA_OUT)

# Bloc 1 — Table de liaison commune → bassin de vie

**Contributeurs :** Gabrielle RICHERT · **Date :** 14/10/2026
 
## 1.1. table de liaison commune à bassin de vie
### Pourquoi cette table

La maille retenue pour le projet est le **bassin de vie 2022** (Insee). Or chaque source donne un code commune, et ce code n'est pas toujours celui que connaît le fichier des bassins de vie :

- **Les communes fusionnées.** Le fichier des bassins de vie est en géographie au 1er janvier 2026. DVF+ garde le code de la commune à la date de la vente : une commune qui a fusionné depuis a disparu du fichier des bassins.
- **Paris, Lyon et Marseille.** DVF+ les code par arrondissement (75101, 69381, 13201…), alors que le fichier des bassins ne connaît que la ville entière (75056, 69123, 13055).

Le test de jointure entre la table cible DVF+ et le fichier des bassins l'a montré : **6,4 % des ventes ne trouvaient pas de bassin**, dont 6,0 % pour les arrondissements et 0,4 % pour les communes fusionnées.

### Le principe

On construit **une seule table pour toute l'équipe** : une ligne par code commune ayant existé depuis 2003, avec son code en 2026 et son bassin de vie. Chaque source s'y rattache par son propre code commune, quelle que soit sa géographie (2024 pour le recensement, 2026 pour la délinquance, l'année de la vente pour DVF+). C'est possible parce que la table de passage de l'Insee contient aussi les communes actuelles, qui pointent vers elles-mêmes.

On utilise deux fichiers Insee :
- `table_passage_geo2003_geo2026.xlsx` : chaque commune ayant existé depuis 2003 et son code en 2026 ;
- `BV2022_2022_2026.xlsx` : chaque commune de 2026 et son bassin de vie 2022 (onglet `Composition_communale`), et la description des bassins (onglet `BV2022`).

### Lire les deux fichiers Insee

Dans les fichiers Excel de l'Insee, les 5 premières lignes sont des titres : les noms de colonnes sont sur la 6e ligne (`skiprows=5`).

`dtype=str` lit tous les codes en texte. Sinon, pandas les lit comme des nombres : `01001` devient `1001`, et les codes de la Corse (`2A004`) posent problème.

In [2]:
# Table de passage : une ligne par commune ayant existé depuis 2003
passage = pd.read_excel(DATA_RAW / "table_passage_geo2003_geo2026.xlsx",
                        sheet_name="Table de passage", skiprows=5, dtype=str)

# Bassins de vie : composition communale et description des bassins
fichier_bv = DATA_RAW / "BV2022_2022_2026.xlsx"
compo = pd.read_excel(fichier_bv, sheet_name="Composition_communale", skiprows=5, dtype=str)
bassins = pd.read_excel(fichier_bv, sheet_name="BV2022", skiprows=5, dtype=str)

print("Table de passage    :", passage.shape)
print("Composition         :", compo.shape)
print("Bassins de vie      :", bassins.shape)
passage.head()

Table de passage    : (36743, 3)
Composition         : (34875, 7)
Bassins de vie      : (1707, 4)


,CODGEO_INI,CODGEO_2026,LIBGEO_2026
0,01001,01001,L'Abergement-Clémenciat
1,01002,01002,L'Abergement-de-Varey
2,01004,01004,Ambérieu-en-Bugey
3,01005,01005,Ambérieux-en-Dombes
4,01006,01006,Ambléon


### Ajouter les arrondissements de Paris, Lyon et Marseille

La table de passage ne contient pas les arrondissements : on les ajoute à la main. Chaque arrondissement pointe vers le code de sa ville en 2026 :
- Paris : 75101 à 75120 → 75056 (20 arrondissements) ;
- Lyon : 69381 à 69389 → 69123 (9) ;
- Marseille : 13201 à 13216 → 13055 (16).

Dans la colonne `LIBGEO_2026`, on met le nom de la ville : c'est le libellé de la commune en 2026, comme pour les autres lignes de la table.

In [3]:
# Code de la ville en 2026 : (nom de la ville, codes de ses arrondissements)
villes = {
    "75056": ("Paris", range(75101, 75121)),
    "69123": ("Lyon", range(69381, 69390)),
    "13055": ("Marseille", range(13201, 13217)),
}

lignes = []
for code_ville, (nom, arrondissements) in villes.items():
    for code_arr in arrondissements:
        lignes.append({"CODGEO_INI": str(code_arr), "CODGEO_2026": code_ville, "LIBGEO_2026": nom})

passage = pd.concat([passage, pd.DataFrame(lignes)], ignore_index=True)

# Contrôle : 45 lignes ajoutées (20 + 9 + 16)
print("Lignes ajoutées :", len(lignes))
print("Table de passage :", passage.shape)

Lignes ajoutées : 45
Table de passage : (36788, 3)


### Une seule ligne par code : les communes scindées

Quand une commune a été **scindée**, son ancien code renvoie vers plusieurs communes de 2026. Exemple : Neussargues en Pinatelle (15141), créée en 2017 par fusion, a été redivisée en 2025 entre Celles, Chalinargues, Chavagnac et Neussargues-Moissac. Le code 15141 a donc 4 lignes dans la table de passage.

Pour une jointure, il faut **une seule ligne par code** : sinon une vente portant ce code serait comptée 4 fois. On applique la règle de l'équipe :
- on garde la ligne où le code pointe vers lui-même (15141 → 15141) ;
- seul le code 52427 n'a pas de ligne vers lui-même : on garde la ligne vers La Porte du Der (52331).

Ces scissions concernent 23 codes et pèsent très peu de ventes.

In [4]:
# Codes qui ont plusieurs lignes (communes scindées)
scindee = passage["CODGEO_INI"].duplicated(keep=False)

# La ligne où le code pointe vers lui-même
identite = passage["CODGEO_INI"] == passage["CODGEO_2026"]

# Le cas particulier 52427 -> 52331
cas_52427 = (passage["CODGEO_INI"] == "52427") & (passage["CODGEO_2026"] == "52331")

print("Codes scindés :", passage.loc[scindee, "CODGEO_INI"].nunique())

# On garde : les codes non scindés, et pour les codes scindés la ligne retenue par la règle
passage = passage[~scindee | identite | cas_52427]

# Contrôle : chaque code n'apparaît plus qu'une fois
print("Un code = une ligne :", passage["CODGEO_INI"].is_unique)
print("Table de passage :", passage.shape)

Codes scindés : 23
Un code = une ligne : True
Table de passage : (36759, 3)


### Ajouter le bassin de vie

On rattache à chaque ligne le bassin de vie de sa commune de 2026, puis la description du bassin (libellé et type).

L'option `validate="many_to_one"` contrôle la jointure : plusieurs anciens codes peuvent pointer vers le même bassin (*many*), mais chaque commune de 2026 ne doit avoir qu'un seul bassin (*one*). Si ce n'est pas le cas, pandas s'arrête avec une erreur au lieu de dupliquer des lignes sans prévenir.

Le type de bassin vaut 1 (rural), 2 (intermédiaire) ou 3 (dense).

In [5]:
# Bassin de vie de chaque commune de 2026
liaison = passage.merge(compo[["CODGEO", "BV2022"]], left_on="CODGEO_2026", right_on="CODGEO",
                        how="left", validate="many_to_one").drop(columns="CODGEO")

# Description du bassin : libellé et type
liaison = liaison.merge(bassins[["BV2022", "LIBBV2022", "TYPE_BV2022"]], on="BV2022",
                        how="left", validate="many_to_one")

print(liaison.shape)
liaison.head()

(36759, 6)


,CODGEO_INI,CODGEO_2026,LIBGEO_2026,BV2022,LIBBV2022,TYPE_BV2022
0,01001,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,2
1,01002,01002,L'Abergement-de-Varey,01004,Ambérieu-en-Bugey,2
2,01004,01004,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,2
3,01005,01005,Ambérieux-en-Dombes,01333,Saint-André-de-Corcy,2
4,01006,01006,Ambléon,01034,Belley,2


### Contrôler et enregistrer

Trois contrôles avant d'enregistrer :
- chaque code n'apparaît qu'une fois ;
- chaque code a un bassin de vie ;
- les arrondissements sont bien rattachés au bassin de leur ville.

La table est enregistrée en Parquet : contrairement au CSV, ce format garde les codes en texte à la relecture.

In [6]:
print("Lignes :", len(liaison))
print("Un code = une ligne :", liaison["CODGEO_INI"].is_unique)
print("Codes sans bassin :", liaison["BV2022"].isna().sum())

# Arrondissements : un seul bassin par ville attendu
arrondissements = liaison["CODGEO_INI"].str.match(r"^(751|6938|132)\d")
print(liaison[arrondissements].groupby("CODGEO_2026")[["BV2022", "LIBBV2022"]].nunique())

liaison.to_parquet(DATA_OUT / "liaison_commune_bv2022.parquet", index=False)

Lignes : 36759
Un code = une ligne : True
Codes sans bassin : 0
             BV2022  LIBBV2022
CODGEO_2026                   
13055             1          1
69123             1          1
75056             1          1


### Résultat du bloc 1.1

- `data/processed/liaison_commune_bv2022.parquet` : une ligne par code commune ayant existé depuis 2003, plus les 45 arrondissements de Paris, Lyon et Marseille.
- Colonnes : `CODGEO_INI` (code d'origine, clé de jointure), `CODGEO_2026`, `LIBGEO_2026`, `BV2022`, `LIBBV2022`, `TYPE_BV2022`.
- **36 759 lignes** : 36 743 codes de la table de passage, plus 45 arrondissements, moins 29 lignes en trop pour les 23 communes scindées.
- Codes sans bassin : … *(à compléter après exécution)*

**Comment l'utiliser :** toute source se rattache par son code commune, sur la colonne `CODGEO_INI`, avec `validate="many_to_one"`. Exemple pour DVF+ : `ventes.merge(liaison, left_on="l_codinsee", right_on="CODGEO_INI", how="left", validate="many_to_one")`.

**Attention :** une source qui donne à la fois la ville entière et ses arrondissements (la délinquance, par exemple) compterait deux fois Paris, Lyon et Marseille. Il faut n'en garder qu'un niveau avant la jointure.

## 1.2. Scindage des plus gros bassins de vie pour obtention de bassins de vie plus homogènes sur le territoire

### Principe
Le bassin de vie de Paris prends en compte plus de 400 communes et ainsi les données uniquement sur ce bassin de vie sont très hétégogènes. Il faut donc diviser ces bassins de vie (Paris, Lyon, Marseille notamment). Pour les scinder il a été décidé en groupe de partir de l'information du nombre d'habitants par bassin de vie.

### Pourquoi filtrer sur la population ?
Filtrer sur la population totale du BV au lieu de sa densité moyenne s'explique par deux raisons majeures de géographie urbaine :

1. L'effet de dilution administrative (le piège de la moyenne) :
La densité moyenne d'un bassin de vie dépend des frontières administratives des communes rurales qui ont été rattachées autour de la ville-centre.

2. La masse critique du marché immobilier :
Un volume élevé d'habitants (ex. $> 100\ 000\text{ hab.}$) garantit un réseau urbain structuré avec de réels sous-marchés (un cœur hyper-dense vs une banlieue pavillonnaire vs du rural éloigné).

**En résumé :** On utilise la population totale pour cibler les grandes agglomérations, puis la densité communale/infra-communale pour découper l'intérieur de ces agglomérations.

### Récupération de la table INSEE des informations sur les communes

Puis calcul de la population par bassin de vie.

In [ ]:
df_communes = pd.read_csv(DATA_RAW / "communes-france-2026.csv",
    dtype={'code_insee': str, 'code_bassin_de_vie': str, 'grille_densite': int})

# Harmonisation du code INSEE sur 5 caractères
df_communes['code_insee'] = df_communes['code_insee'].str.zfill(5)

# Calcul de la population totale par bassin de vie
df_communes['pop_bv_totale'] = df_communes.groupby('code_bassin_de_vie')['population'].transform('sum')

### Conditions de scissions

Pour les bassins de vie < 100 000 habitants, on conserve le bassin de vie existants.

Pour les bassins de vie > 100 000 habitants, on les scinde en fonction de leur densité de population : les zones les plus denses (grille densité de 1 ou 2), on leur attribue le suffixe _1, pour les zones moins denses le suffixe _2 et pour les zones rurales qui étaient initialement contenues dans le bassin de vie initial, le suffixe _3 dans la colonne sous_bassin_de_vie.

In [ ]:
# Attribution vectorisée des sous-bassins
conditions = [
    df_communes['pop_bv_totale'] < 100000,
    df_communes['grille_densite'].isin([1, 2]),
    df_communes['grille_densite'].isin([3, 4]),
]
choix = [
    df_communes['code_bassin_de_vie'],
    df_communes['code_bassin_de_vie'] + '_1',
    df_communes['code_bassin_de_vie'] + '_2',
]

df_communes['sous_bassin_de_vie'] = np.select(
    conditions, choix, default=df_communes['code_bassin_de_vie'] + '_3'
)

### Traitement des arrondissements

On doit traiter à part les arrondissements des 3 agglomérations car chacun possède suffisamment d'habitants pour être considéré comme un bassin de vie à part entière.

In [ ]:
pop_arrondissements = {
    # Paris (75101 à 75120)
    '75101': 15919,
    '75102': 21130,
    '75103': 34120,
    '75104': 28088,
    '75105': 57380,
    '75106': 39625,
    '75107': 47970,
    '75108': 35190,
    '75109': 59250,
    '75110': 83450,
    '75111': 142500,
    '75112': 140100,
    '75113': 177800,
    '75114': 133400,
    '75115': 227700,
    '75116': 162200,
    '75117': 163900,
    '75118': 188500,
    '75119': 181600,
    '75120': 189800,
    # Lyon (69381 à 69389)
    '69381': 29500,
    '69382': 30900,
    '69383': 101700,
    '69384': 35700,
    '69385': 48800,
    '69386': 52100,
    '69387': 82600,
    '69388': 86300,
    '69389': 51500,
    # Marseille (13201 à 13216)
    '13201': 39500,
    '13202': 23600,
    '13203': 45100,
    '13204': 49100,
    '13205': 45800,
    '13206': 42700,
    '13207': 34800,
    '13208': 82100,
    '13209': 76400,
    '13210': 56500,
    '13211': 57300,
    '13212': 61200,
    '13213': 92300,
    '13214': 61800,
    '13215': 79400,
    '13216': 38400,
}

# Mapping Arrondissement -> Commune Parente
parent_map = {}
for code in pop_arrondissements:
    if code.startswith('751'):
        parent_map[code] = '75056'
    elif code.startswith('6938'):
        parent_map[code] = '69123'
    elif code.startswith('132'):
        parent_map[code] = '13055'

lignes_arrondissements = []
for code_arm, pop_arm in pop_arrondissements.items():
    code_parent = parent_map[code_arm]
    parent_matches = df_communes[df_communes['code_insee'] == code_parent]

    if not parent_matches.empty:
        parent_row = parent_matches.iloc[0].copy()
        parent_row['code_insee'] = code_arm

        if 'nom_standard' in parent_row:
            parent_row['nom_standard'] = (
                f"{parent_row['nom_standard']} Arrondissement {code_arm[-2:]}"
            )

        parent_row['population'] = pop_arm
        # Le sous-bassin de l'arrondissement prend son propre code INSEE
        parent_row['sous_bassin_de_vie'] = code_arm
        lignes_arrondissements.append(parent_row)

df_bv_scindes = pd.concat(
    [df_communes, pd.DataFrame(lignes_arrondissements)], ignore_index=True
)


### Jointure avec la table de liaison précédente

Attention aux arrondissements, il est important d'utiliser une clé qui détectera la présence des arrondissements pour les prendre en compte.

In [ ]:
df_liaison = pd.read_parquet(DATA_OUT / "liaison_commune_bv2022.parquet")

#Clé qui permet de garder le code arrondissement quand c'en est un
arrondissement = df_liaison["CODGEO_INI"].str.match(r"^(751\d\d|6938\d|132\d\d)$")
df_liaison["code_jointure"] = df_liaison["CODGEO_2026"].where(~arrondissement, df_liaison["CODGEO_INI"])

# Formater les codes INSEE sur 5 caractères
df_liaison['CODGEO_2026'] = df_liaison['CODGEO_2026'].str.zfill(5)
df_bv_scindes['code_insee_clean'] = df_bv_scindes['code_insee'].str.zfill(5)

# Fusion : liaison de l'ancien/actuel code vers le référentiel 2026 via CODGEO_2026
df_merged = pd.merge(
    df_liaison,
    df_bv_scindes[
        ['code_insee_clean', 'sous_bassin_de_vie', 'pop_bv_totale']
    ].drop_duplicates(),
    left_on='code_jointure',
    right_on='code_insee_clean',
    how='left', validate="many_to_one"
)

# Nettoyage des colonnes
df_merged.drop(columns=['code_insee_clean','LIBGEO_2026','LIBBV2022', 'code_jointure'], inplace=True)

# Communes absentes de communes-france-2026 (sans grille de densité) :
# si leur bassin n'est pas scindé, le sous-bassin est le bassin lui-même
non_scindes = df_merged.loc[df_merged["sous_bassin_de_vie"] == df_merged["BV2022"], "BV2022"].unique()
manquant = df_merged["sous_bassin_de_vie"].isna() & df_merged["BV2022"].isin(non_scindes)
df_merged.loc[manquant, "sous_bassin_de_vie"] = df_merged.loc[manquant, "BV2022"]
print("Sous-bassins complétés par le bassin non scindé :", manquant.sum())

#4. Contrôles et enregistrement
print("Codes sans sous-bassin :", df_merged["sous_bassin_de_vie"].isna().sum()) #doit afficher 0
print(df_merged.loc[df_merged["CODGEO_INI"].isin(["75101", "69381", "13201"]), ["CODGEO_INI", "sous_bassin_de_vie"]]) # 75101, 69381 et 13201 doivent être des sous bassins
print("Nombre de territoires :", df_merged["sous_bassin_de_vie"].nunique())

df_merged.to_parquet(DATA_OUT / "liaison_bv_scindes.parquet", index=False)

### Superficie des communes 2026
**Contributeurs :** Christian & Cécile · **Date :** 03/11/2026

Superficie de chaque commune en km² (`communes-france-2026.csv`), pour la densité de population calculée au feature engineering.

Paris, Lyon et Marseille n'y figurent qu'en ville entière, alors que leurs arrondissements sont des sous-bassins à part. La ville entière est donc **remplacée par ses 45 arrondissements**, avec leur superficie réelle : réponse de l'API Découpage administratif (`superficie_arrondissements_geoapi.json`, source `superficie_arrondissements` du téléchargement), en hectares, convertie en km².

Contrôles : 45 arrondissements, surfaces positives, et somme des arrondissements proche de la superficie de la ville (moins de 10 % d'écart). Ce dernier contrôle détecte une erreur d'unité (hectares au lieu de km²), pas une imprécision.

In [ ]:
# --- Superficie de chaque commune 2026 (km²), pour la densité calculée au feature engineering ---
# Paris, Lyon, Marseille : le référentiel ne donne que la ville entière. Elle est remplacée par ses
# 45 arrondissements, avec leur superficie réelle (API Découpage administratif, en hectares)
superficie = pd.read_csv(DATA_RAW / "communes-france-2026.csv", usecols=["code_insee", "superficie_km2"],
                         dtype={"code_insee": str})
superficie["code_insee"] = superficie["code_insee"].str.zfill(5)
superficie = superficie.rename(columns={"code_insee": "code_commune"})

arrondissements = pd.read_json(DATA_RAW / "superficie_arrondissements_geoapi.json", dtype={"code": str})
arrondissements = arrondissements.rename(columns={"code": "code_commune"})
arrondissements["superficie_km2"] = arrondissements["surface"] / 100      # 1 km² = 100 hectares
arrondissements = arrondissements[["code_commune", "superficie_km2"]]

# Contrôles : 45 arrondissements, surfaces positives, somme proche de la superficie de la ville
# (moins de 10 % d'écart : détecte une erreur d'unité, pas une imprécision)
VILLES_ARRONDISSEMENTS = {"75056": r"^751\d\d$", "69123": r"^6938\d$", "13055": r"^132\d\d$"}
assert len(arrondissements) == 45 and arrondissements["code_commune"].is_unique, "45 arrondissements uniques attendus"
assert (arrondissements["superficie_km2"] > 0).all(), "Superficie d'arrondissement nulle ou vide"
for ville, motif in VILLES_ARRONDISSEMENTS.items():
    somme = arrondissements.loc[arrondissements["code_commune"].str.match(motif), "superficie_km2"].sum()
    sup_ville = superficie.loc[superficie["code_commune"] == ville, "superficie_km2"].iloc[0]
    print(f"Bloc 1 - superficie {ville} : ville {sup_ville:.1f} km² | somme des arrondissements {somme:.1f} km²")
    assert abs(somme / sup_ville - 1) < 0.10, f"{ville} : somme des arrondissements éloignée de la ville (unité ?)"

# Ville entière retirée, arrondissements ajoutés : la superficie n'est comptée qu'une fois
superficie = pd.concat([superficie[~superficie["code_commune"].isin(VILLES_ARRONDISSEMENTS)], arrondissements],
                       ignore_index=True)
print("Bloc 1 - superficie : communes et arrondissements :", len(superficie),
      "| un code = une ligne :", superficie["code_commune"].is_unique,
      "| sans superficie :", superficie["superficie_km2"].isna().sum())
superficie.to_parquet(DATA_OUT / "communes_superficie_2026.parquet", index=False)

### Résultat du bloc 1.2.

- `data/processed/liaison_bv_scindes.parquet` : une ligne par code commune ayant existé depuis 2003, plus les 45 arrondissements de Paris, Lyon et Marseille redécoupés par arrondissements. Les plus gros bassins de vie ont été scindés en 3 sous bassins en fonction de leur densité de population.
- Colonnes : `CODGEO_INI` (code d'origine, clé de jointure), `CODGEO_2026`, `LIBGEO_2026`, `BV2022`, `TYPE_BV2022`, `sous_bassin_de_vie`, `pop_bv_totale`.
- **36 759 lignes** : 36 743 codes de la table de passage, plus 45 arrondissements, moins 29 lignes en trop pour les 23 communes scindées.
- Codes sans sous-bassin : 0 ; 8 codes complétés par leur bassin non scindé ; 1 933 sous-bassins.
- `data/processed/communes_superficie_2026.parquet` : superficie de chaque commune 2026 en km², Paris, Lyon et Marseille remplacées par leurs 45 arrondissements (superficie réelle), pour la densité calculée au feature engineering.

**Comment l'utiliser :** toute source se rattache par son code commune, sur la colonne `CODGEO_INI`, avec `validate="many_to_one"`. Exemple pour DVF+ : `ventes.merge(liaison, left_on="l_codinsee", right_on="CODGEO_INI", how="left", validate="many_to_one")`.

**Attention :** une source qui donne à la fois la ville entière et ses arrondissements (la délinquance, par exemple) compterait deux fois Paris, Lyon et Marseille. Il faut n'en garder qu'un niveau avant la jointure.

# Bloc 2 — Filtrage de DVF+

**Contributeurs :** Cécile NAUDET · **Date :** 12/10/2026

### Ce que l'audit a montré

L'audit de DVF+ a porté sur un échantillon de 100 000 mutations, tirées dans les 97 fichiers départementaux (16,5 millions de mutations, 2014-2025).

- **Une base hétérogène** : 92 % de ventes, 6,5 % de VEFA, le reste en échanges, adjudications et ventes de terrains à bâtir. Une partie des mutations porte sur des terrains, des locaux d'activité ou plusieurs biens à la fois.
- **Une valeur foncière par mutation, pas par bien.** Le prix ne se rapporte à un bien identifiable que si la mutation porte sur un seul logement.

L'audit a conduit à fixer le **périmètre** :
- ventes classiques uniquement (VEFA exclues) ;
- une seule maison ou un seul appartement, avec d'éventuelles dépendances ;
- une seule commune ;
- France métropolitaine, sans les DOM (l'Alsace-Moselle est absente de DVF+).

### La méthode

Les fichiers sont trop gros pour être lus d'un coup. On lit chaque département **par blocs de 200 000 lignes**, on filtre chaque bloc, puis on assemble les lignes gardées. Les 93 départements filtrés sont ensuite réunis dans **un seul fichier** : `dvf_plus_residentiel.parquet`.

### Deux façons d'obtenir la population filtrée

Ce fichier sert à tout le reste : l'échantillon (bloc 3) et les tables cibles (bloc 4). Il y a deux façons de l'obtenir :
- **le parcours complet** : télécharger DVF+ brut (`python 00_telechargement/telechargement.py dvf`, plusieurs Go) puis exécuter ce bloc ;
- **le parcours rapide** : télécharger directement le fichier filtré depuis le Drive (`python 00_telechargement/telechargement.py dvf_residentiel`, environ 1,4 Go).

Le bloc vérifie d'abord si le fichier existe déjà. Si oui, il ne refait pas le filtrage. Pour le forcer, il suffit de supprimer le fichier.

### Vérifier si la population filtrée existe déjà, et lister les fichiers à traiter

DVF+ est livré en 97 fichiers CSV, un par département. On écarte les 4 DOM : il reste 93 fichiers.

In [7]:
fichier_pop = DATA_OUT / "dvf_plus_residentiel.parquet"
dossier_dep = DATA_OUT / "dvf_plus_residentiel_departements"   # fichiers temporaires, un par département

deja_present = fichier_pop.exists()

if deja_present:
    print("Population déjà présente :", fichier_pop.name, "-> le filtrage n'est pas refait")
else:
    # Fichiers départementaux, sans les DOM (hors périmètre)
    DOM = {"dvf_plus_d971", "dvf_plus_d972", "dvf_plus_d973", "dvf_plus_d974"}
    fichiers = [f for f in sorted((DATA_RAW / "DVF+").glob("dvf_plus_d*.csv")) if f.stem not in DOM]
    # Contrôle : 93 fichiers attendus (97 - 4 DOM)
    print("Nombre de fichiers retenus :", len(fichiers))

Population déjà présente : dvf_plus_residentiel.parquet -> le filtrage n'est pas refait


### Écrire les règles de filtrage

Une fonction regroupe toutes les règles du périmètre. On l'applique ensuite à chaque bloc de lignes.

In [8]:
def filtre_perimetre(df):
    """Garde les ventes d'un seul logement ancien, sur une seule commune."""
    # Vente classique uniquement (la VEFA a son propre libellé)
    nature_ok = df["libnatmut"] == "Vente"
    # Une seule commune, pour que l_codinsee contienne un code unique
    une_commune = df["nbcomm"] == 1
    # Une seule maison, sans appartement ni local d'activité (dépendances acceptées)
    maison_ok = (
        (df["libtypbien"] == "UNE MAISON")
        & (df["nblocmai"] == 1)
        & (df["nblocapt"] == 0)
        & (df["nblocact"] == 0)
    )
    # Un seul appartement, sans maison ni local d'activité (dépendances acceptées)
    appartement_ok = (
        (df["libtypbien"] == "UN APPARTEMENT")
        & (df["nblocapt"] == 1)
        & (df["nblocmai"] == 0)
        & (df["nblocact"] == 0)
    )
    return df.loc[nature_ok & une_commune & (maison_ok | appartement_ok)]

### Filtrer chaque département

Pour chaque département : lecture par blocs, filtrage, assemblage, enregistrement dans un fichier temporaire. Les codes géographiques (`coddep`, `l_codinsee`) sont lus en texte pour garder le zéro initial.

Un bilan compte, pour chaque département, les mutations lues et les ventes gardées. Il sert à vérifier le filtrage et à le décrire dans le rapport.

*Durée : quelques minutes.*

In [9]:
import time

if not deja_present:
    dossier_dep.mkdir(exist_ok=True)
    types_texte = {"coddep": str, "l_codinsee": str}   # codes géographiques lus en texte
    bilan = []            # une ligne par département : lignes brutes et retenues
    debut = time.time()   # chronomètre

    for fichier in fichiers:
        morceaux = []
        n_brut = 0
        # Lecture par blocs pour limiter la mémoire utilisée
        for bloc in pd.read_csv(fichier, sep="|", chunksize=200_000,
                                dtype=types_texte, low_memory=False):
            n_brut += len(bloc)
            morceaux.append(filtre_perimetre(bloc))
        pop = pd.concat(morceaux, ignore_index=True)
        # Un fichier temporaire par département
        pop.to_parquet(dossier_dep / f"{fichier.stem}.parquet", index=False)
        bilan.append({"fichier": fichier.stem, "brut": n_brut, "retenu": len(pop)})

    # Tableau récapitulatif, conservé pour le rapport
    bilan = pd.DataFrame(bilan)
    bilan["part_retenue_%"] = (bilan["retenu"] / bilan["brut"] * 100).round(1)
    bilan.to_csv(DATA_OUT / "bilan_filtrage.csv", index=False)

    print("Durée :", round((time.time() - debut) / 60, 1), "minutes")
    print(bilan[["brut", "retenu"]].sum())

### Réunir les 93 départements dans un seul fichier

Deux difficultés :
- **Les types de colonnes ne sont pas toujours les mêmes d'un département à l'autre.** Une colonne de texte entièrement vide dans un petit département est lue comme une colonne de nombres. Les deux versions ne peuvent pas être réunies telles quelles. On fixe donc un type commun pour chaque colonne : du texte si au moins un département contient du texte, sinon un nombre décimal si les types diffèrent.
- **Les 9,7 millions de lignes, avec toutes leurs colonnes, ne tiennent pas facilement en mémoire.** On écrit donc le fichier unique **département par département**, avec un `ParquetWriter` de la bibliothèque `pyarrow` (celle qu'utilise pandas pour le format Parquet).

Une fois le fichier unique écrit, les fichiers temporaires par département sont supprimés.

In [10]:
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

if not deja_present:
    parquets_dep = sorted(dossier_dep.glob("*.parquet"))

    # 1. Type commun de chaque colonne, en lisant seulement les schémas (pas les données)
    types = {}
    for p in parquets_dep:
        for champ in pq.read_schema(p):
            ancien = types.get(champ.name)
            if ancien is None or ancien == champ.type:
                types[champ.name] = champ.type
            elif pa.types.is_large_string(ancien) or pa.types.is_large_string(champ.type) \
                    or pa.types.is_string(ancien) or pa.types.is_string(champ.type):
                types[champ.name] = pa.large_string()   # du texte d'un côté : texte
            else:
                types[champ.name] = pa.float64()        # deux types de nombres : décimal
    colonnes = pq.read_schema(parquets_dep[0]).names       # ordre des colonnes du premier fichier
    schema = pa.schema([(c, types[c]) for c in colonnes])

    # 2. Écriture département par département dans un seul fichier
    with pq.ParquetWriter(fichier_pop, schema) as fichier_unique:
        for p in parquets_dep:
            table = pq.read_table(p).select(colonnes).cast(schema)
            fichier_unique.write_table(table)

    # 3. Suppression des fichiers temporaires
    shutil.rmtree(dossier_dep)
    print("Fichier unique écrit :", fichier_pop.name)

### Contrôler la population obtenue

Trois contrôles :
- le nombre et la taille des fichiers produits ;
- le zéro initial des codes, sur l'Ain (01) ;
- les départements où la part du résidentiel est la plus faible et la plus forte.

In [11]:
print("Taille :", round(fichier_pop.stat().st_size / 1e9, 2), "Go")
print("Ventes :", pq.ParquetFile(fichier_pop).metadata.num_rows)

# Zéro devant les codes : contrôle sur l'Ain (01), en ne lisant que ce département et 2 colonnes
d01 = pd.read_parquet(fichier_pop, columns=["coddep", "l_codinsee"], filters=[("coddep", "==", "01")])
print(d01.head(3))

print("Départements :", pd.read_parquet(fichier_pop, columns=["coddep"])["coddep"].nunique())

# Bilan du filtrage : seulement si le filtrage vient d'être fait
if not deja_present:
    print(bilan.sort_values("part_retenue_%").head(5))
    print(bilan.sort_values("part_retenue_%").tail(5))

Taille : 1.4 Go
Ventes : 9680495
  coddep l_codinsee
0     01      01378
1     01      01160
2     01      01022
Départements : 93


### Résultat du bloc 2

- 16 375 738 mutations brutes → **9 680 495 ventes retenues (59,1 %)**, en 93 fichiers Parquet (1,4 Go) dans `data/processed/dvf_plus_residentiel/`.
- `data/processed/bilan_filtrage.csv` : lignes brutes et retenues par département.
- Les codes géographiques restent en texte : le zéro initial est conservé (`01`, `01378`).
- La part du résidentiel varie du simple au double selon les départements : 35,8 % en Lozère, 71,9 % à Paris. C'est ce qui oblige à filtrer avant de tirer l'échantillon.

# Bloc 3 — Échantillon d'apprentissage (2014-2024)

**Contributeurs :** Cécile NAUDET · **Date :** 14/10/2026, mis à jour le 22/10/2026

### Pourquoi un nouvel échantillon

L'échantillon de l'audit (100 000 mutations) ne convient pas pour la suite :
- **Il a été tiré avant le filtrage.** Une fois réduit au résidentiel, il laisse trop peu de ventes par département et par année : la Lozère n'avait que 150 mutations au total, tous types confondus.
- **La part du résidentiel varie fortement selon les départements** (35,8 % en Lozère, 71,9 % à Paris). Tirer avant de filtrer surreprésenterait les départements ruraux. Il faut filtrer d'abord, puis tirer.
- **Il ne respecte pas le périmètre retenu** : il contient les VEFA, les DOM et les mutations portant sur plusieurs communes.

### La méthode : un tirage stratifié

On découpe la population en **strates** : une strate = un département × une année. Dans chaque strate, on tire **une vente sur dix**, avec **au moins 300 ventes** pour que les petits départements restent étudiables.

L'échantillon porte sur **2014-2024**, les années d'apprentissage. L'année 2025 est réservée au test : aucune vente de 2025 n'est tirée.

### Compter les ventes par strate

On ne lit que les deux colonnes qui définissent la strate : le département et l'année. Toute la population tient alors en mémoire.

In [12]:
import numpy as np
import pyarrow.parquet as pq

# Population filtrée par le bloc 2
fichier_pop = DATA_OUT / "dvf_plus_residentiel.parquet"

# Nombre de ventes par strate (département × année)
# value_counts compte chaque combinaison (département, année)
strates = (pd.read_parquet(fichier_pop, columns=["coddep", "anneemut"])
             .value_counts().rename("N_h").reset_index())

print("Nombre de strates :", len(strates))
print(strates.groupby("anneemut")["N_h"].sum())

Nombre de strates : 1116
anneemut
2014     592452
2015     673224
2016     718588
2017     819528
2018     818058
2019     904755
2020     881175
2021    1022405
2022     978012
2023     773771
2024     703296
2025     795231
Name: N_h, dtype: int64


### Calculer combien tirer dans chaque strate

Pour chaque strate *h* :
- `N_h` est le nombre de ventes de la strate ;
- `n_h` est le nombre de ventes à tirer : 10 % de `N_h`, relevé à 300 si c'est moins, sans dépasser `N_h` ;
- `w_h = N_h / n_h` est le **poids** : le nombre de ventes de la population que représente chaque vente tirée.

Le plancher de 300 fait que l'échantillon n'est plus proportionnel : les petits départements y sont surreprésentés. Le poids `w_h` corrige cet écart dans les calculs.

In [13]:
# Taux de tirage et plancher par strate
taux, plancher = 0.10, 300

app = strates[strates["anneemut"] <= 2024].copy()

# Part proportionnelle (10 %), relevée au plancher, plafonnée à la taille de la strate
app["n_h"] = np.minimum(app["N_h"], np.maximum(plancher, np.round(app["N_h"] * taux))).astype(int)
app["w_h"] = app["N_h"] / app["n_h"]

print("Population 2014-2024 :", app["N_h"].sum())
print("Taille de l'échantillon :", app["n_h"].sum())
print("Strates relevées par le plancher :", (app["n_h"] > np.round(app["N_h"] * taux)).sum())
print("Poids : de", round(app["w_h"].min(), 2), "à", round(app["w_h"].max(), 2))

Population 2014-2024 : 8885264
Taille de l'échantillon : 907306
Strates relevées par le plancher : 209
Poids : de 1.83 à 10.02


### Tirer l'échantillon

Pour chaque vente, on tire un nombre au hasard (un « ticket de loterie »). Dans chaque strate, on garde les `n_h` ventes qui ont les plus petits tickets.

La **graine** du générateur (42) est fixée : le même tirage se refait à l'identique à chaque exécution.

In [14]:
# Générateur de nombres aléatoires, graine fixée pour la reproductibilité
rng = np.random.default_rng(42)
morceaux = []

for dep in sorted(app["coddep"].unique()):
    # Les ventes d'un seul département
    d = pd.read_parquet(fichier_pop, filters=[("coddep", "==", dep)])
    # Années d'apprentissage uniquement
    d = d[d["anneemut"] <= 2024]
    # On ajoute à chaque vente le n_h et le w_h de sa strate
    d = d.merge(app[["coddep", "anneemut", "n_h", "w_h"]], on=["coddep", "anneemut"])
    # Ticket de loterie : un nombre au hasard par vente
    d["alea"] = rng.random(len(d))
    # Rang du ticket dans sa strate (1 = plus petit numéro)
    d["rang"] = d.groupby(["coddep", "anneemut"])["alea"].rank(method="first")
    # On garde les n_h premiers rangs de chaque strate
    morceaux.append(d[d["rang"] <= d["n_h"]].drop(columns=["alea", "rang", "n_h"]))

ech = pd.concat(morceaux, ignore_index=True)

# Contrôles : taille totale, respect de n_h dans chaque strate, identifiants uniques
print("Taille de l'échantillon :", len(ech))
verif = ech.groupby(["coddep", "anneemut"]).size().rename("tire").reset_index()
verif = verif.merge(app[["coddep", "anneemut", "n_h"]], on=["coddep", "anneemut"])
print("Strates où le nombre tiré diffère de n_h :", (verif["tire"] != verif["n_h"]).sum())
print("idmutation uniques :", ech["idmutation"].is_unique)

Taille de l'échantillon : 907306
Strates où le nombre tiré diffère de n_h : 0
idmutation uniques : True


### Enregistrer l'échantillon

On retire les 11 colonnes que l'audit a jugées vides, constantes, redondantes ou inutiles pour le modèle. Elles restent dans la population (`dvf_plus_residentiel/`) et peuvent être rattachées par `idmutation`. Ce retrait fait passer le fichier de 124 Mo à 33 Mo.

On enregistre aussi la liste des identifiants tirés, qui permet de retrouver l'échantillon sans le fichier complet.

In [15]:
# Colonnes non conservées à l'issue de l'audit
a_retirer = ["idmutinvar", "idopendata", "codservch", "refdoc", "nbartcgi", "l_artcgi",
             "l_section", "l_idpar", "l_idparmut", "l_idlocmut", "l_dcnt"]
ech = ech.drop(columns=a_retirer)

chemin_ech = DATA_OUT / "dvf_plus_echantillon_apprentissage.parquet"
chemin_ids = DATA_OUT / "ids_echantillon_apprentissage.csv"
ech.to_parquet(chemin_ech, index=False)
ech[["idmutation"]].to_csv(chemin_ids, index=False)

print("Dimensions :", ech.shape)
print("Échantillon :", round(chemin_ech.stat().st_size / 1e6, 1), "Mo")

Dimensions : (907306, 53)
Échantillon : 36.3 Mo


### Vérifier que l'échantillon pondéré est représentatif

On compare la population 2014-2024 et l'échantillon sur deux indicateurs : la part de maisons et le prix médian au m². L'échantillon est calculé deux fois : sans poids, puis avec les poids `w_h`.

Pour la médiane pondérée, chaque vente compte autant que son poids : on trie les prix, on additionne les poids au fur et à mesure, et la médiane est le prix où l'on atteint la moitié du poids total.

In [16]:
def mediane_ponderee(valeurs, poids):
    """Médiane où chaque valeur compte autant que son poids."""
    v = np.asarray(valeurs)
    w = np.asarray(poids)
    ordre = np.argsort(v)                 # rang des valeurs, de la plus petite à la plus grande
    v, w = v[ordre], w[ordre]
    cumul = np.cumsum(w)                  # poids additionnés au fur et à mesure
    return v[np.searchsorted(cumul, cumul[-1] / 2)]   # valeur où l'on atteint la moitié du poids total

cols = ["anneemut", "libtypbien", "valeurfonc", "sbati"]

# Population complète 2014-2024 (mêmes années que l'échantillon), en ne lisant que les colonnes utiles
popu = pd.read_parquet(fichier_pop, columns=cols)
popu = popu[popu["anneemut"] <= 2024]

# Échantillon relu depuis le fichier enregistré (vérifie au passage le fichier lui-même)
e = pd.read_parquet(DATA_OUT / "dvf_plus_echantillon_apprentissage.parquet", columns=cols + ["w_h"])

# Prix au m², en écartant les valeurs foncières manquantes
popu = popu.dropna(subset=["valeurfonc"])
popu["prix_m2"] = popu["valeurfonc"] / popu["sbati"]
e = e.dropna(subset=["valeurfonc"])
e["prix_m2"] = e["valeurfonc"] / e["sbati"]

comparaison = pd.DataFrame({
    "population": [
        (popu["libtypbien"] == "UNE MAISON").mean() * 100,
        popu["prix_m2"].median(),
    ],
    "échantillon sans poids": [
        (e["libtypbien"] == "UNE MAISON").mean() * 100,
        e["prix_m2"].median(),
    ],
    "échantillon avec poids": [
        np.average(e["libtypbien"] == "UNE MAISON", weights=e["w_h"]) * 100,
        mediane_ponderee(e["prix_m2"], e["w_h"]),
    ],
}, index=["part de maisons (%)", "prix médian au m² (€)"]).round(1)

print(comparaison)

                       population  échantillon sans poids  \
part de maisons (%)          57.3                    57.6   
prix médian au m² (€)      2269.7                  2246.0   

                       échantillon avec poids  
part de maisons (%)                      57.3  
prix médian au m² (€)                  2272.7  


### Résultat du bloc 3

- Population d'apprentissage (2014-2024) : 8 885 264 ventes, réparties en 1 023 strates (93 départements × 11 années).
- 209 strates (20 %) sont relevées par le plancher de 300 ventes : ce sont les départements peu peuplés.
- `data/processed/dvf_plus_echantillon_apprentissage.parquet` : **907 306 ventes**, 53 colonnes, 36,3 Mo.
- `data/processed/ids_echantillon_apprentissage.csv` : la liste des `idmutation` tirés.
- Poids `w_h` compris entre 1,83 (Lozère 2014) et 10,02.
- **Représentativité :** avec les poids, l'échantillon reproduit la population (part de maisons identique, 57,3 % ; prix médian à 0,1 % près, 2 273 € contre 2 270 €). Sans poids, il contient un peu trop de maisons (57,6 %) et sous-estime le prix médian d'environ 1 % (2 246 €), à cause de la surreprésentation des départements peu peuplés.

**À retenir :** l'échantillon n'est pas proportionnel. Toute statistique ou tout modèle portant sur plusieurs départements ou plusieurs années doit utiliser la colonne `w_h`.

# Bloc 4 — Tables cibles : prix médian au m²

**Contributeurs :** Cécile NAUDET · **Date :** 15/10/2026, mis à jour le 31/10/2026

### Ce qu'on prépare

La cible du projet est le **prix médian au m²**. Les tables sont préparées selon :
- **la maille géographique** : le sous-bassin de vie, maille retenue pour la modélisation, le bassin de vie et la commune, qui servent aux analyses exploratoires ;
- **le pas de temps** : le mois, le trimestre ou l'année.

On prépare donc **neuf tables** (3 mailles × 3 pas de temps). Dans chaque table, **une ligne = une zone × une période**, avec pour chaque type de bien (maison, appartement) le nombre de ventes et le prix médian au m².

### Les choix communs aux neuf tables

- **Toutes les ventes retenues** (9,7 millions), pas l'échantillon : chaque zone a ainsi le plus de ventes possible.
- **La médiane plutôt que la moyenne** : quelques ventes très chères suffisent à tirer la moyenne vers le haut.
- **La médiane est toujours recalculée à partir des ventes.** Une médiane ne se moyenne pas : la médiane d'un bassin n'est pas la moyenne des médianes de ses communes.
- **Aucun seuil de ventes n'est appliqué ici.** Le nombre de ventes de chaque case est conservé (`n_maison`, `n_appart`). Le seuil retenu, 20 ventes par bassin, par année et par type de bien (EDA, A2), sera appliqué à la modélisation.
- **Une géographie constante dans le temps.** Pour étudier une série temporelle, une zone doit garder le même code sur toute la période. On utilise donc le code de la commune **en 2026** : les ventes d'une commune qui a fusionné sont regroupées sous le code de la commune nouvelle. Dans les tables par commune, Paris, Lyon et Marseille restent découpées **par arrondissement**, comme dans DVF+, le recensement et LOVAC. Dans les tables par bassin de vie, les arrondissements sont rattachés **au bassin de leur ville** (décision du groupe, EDA A2). Dans les tables par sous-bassin, **chaque arrondissement forme son propre sous-bassin**.

### Charger la population (colonnes utiles seulement)

On ne lit que les colonnes nécessaires : la population complète tient alors en mémoire.

In [17]:
# Seulement les colonnes utiles
colonnes = ["idmutation", "anneemut", "moismut", "l_codinsee", "libtypbien", "valeurfonc", "sbati"]
pop = pd.read_parquet(DATA_OUT / "dvf_plus_residentiel.parquet", columns=colonnes)
print(pop.shape)
pop.head()

(9680495, 7)


,idmutation,anneemut,moismut,l_codinsee,libtypbien,valeurfonc,sbati
0,16,2014,5,01378,UNE MAISON,222300.0,88.0
1,476,2014,2,01160,UNE MAISON,1084000.0,212.0
2,8975,2014,9,01022,UNE MAISON,139000.0,90.0
3,9113,2014,12,01210,UN APPARTEMENT,75700.0,41.0
4,9498,2014,4,01185,UNE MAISON,58000.0,39.0


### Calculer le prix au m² et retirer les valeurs aberrantes

On applique les règles d'exclusion fixées pendant l'EDA :
- la valeur foncière doit être renseignée et positive ;
- le prix au m² doit être compris entre 50 € et 50 000 €.

In [18]:
# Prix au m² de chaque vente
pop["prix_m2"] = pop["valeurfonc"] / pop["sbati"]

avant = len(pop)
pop = pop[
    pop["valeurfonc"].notna()
    & (pop["valeurfonc"] > 0)
    & (pop["prix_m2"] >= 50)
    & (pop["prix_m2"] <= 50000)
].copy()
apres = len(pop)
print(f"Avant : {avant:,} | Après : {apres:,} | Exclues : {avant - apres:,} ({(avant - apres) / avant:.2%})")

# Type de bien en version courte, pour les noms de colonnes
pop["type_bien"] = pop["libtypbien"].map({"UNE MAISON": "maison", "UN APPARTEMENT": "appart"})

Avant : 9,680,495 | Après : 9,655,421 | Exclues : 25,074 (0.26%)


### Rattacher chaque vente à sa commune 2026, à son bassin de vie et à son sous-bassin

On utilise la table de liaison du bloc 1.2 (`liaison_bv_scindes`), sur le code commune de la vente (`l_codinsee`).

- **Commune** : le code 2026, sauf pour les arrondissements de Paris, Lyon et Marseille, qui gardent leur propre code.
- **Bassin de vie** : la colonne `BV2022`. Comme la table rattache chaque arrondissement au code de sa ville, les arrondissements tombent dans le bassin de leur ville.
- **Sous-bassin** : la colonne `sous_bassin_de_vie`. Chaque arrondissement y est son propre sous-bassin.

In [ ]:
liaison = pd.read_parquet(DATA_OUT / "liaison_bv_scindes.parquet")

pop = pop.merge(liaison[["CODGEO_INI", "CODGEO_2026", "BV2022", "sous_bassin_de_vie"]],
                left_on="l_codinsee", right_on="CODGEO_INI", how="left", validate="many_to_one")

# Code commune en géographie constante : code 2026, sauf pour les arrondissements
arrondissement = pop["l_codinsee"].str.match(r"^(751\d\d|6938\d|132\d\d)$")
pop["code_commune"] = pop["CODGEO_2026"].where(~arrondissement, pop["l_codinsee"])

# Contrôles : chaque vente doit avoir une commune, un bassin de vie et un sous-bassin
print("Ventes sans commune     :", pop["code_commune"].isna().sum())
print("Ventes sans bassin      :", pop["BV2022"].isna().sum())
print("Ventes sans sous-bassin :", pop["sous_bassin_de_vie"].isna().sum())
print("Communes (géographie 2026, arrondissements séparés) :", pop["code_commune"].nunique())
print("Bassins de vie (arrondissements dans le bassin de leur ville) :", pop["BV2022"].nunique())
print("Sous-bassins (un par arrondissement) :", pop["sous_bassin_de_vie"].nunique())

### Ajouter les périodes

DVF+ donne l'année (`anneemut`) et le mois (`moismut`) de chaque vente. Le trimestre s'en déduit : les mois 1 à 3 forment le trimestre 1, les mois 4 à 6 le trimestre 2, etc.

In [20]:
pop["annee"] = pop["anneemut"]
pop["mois"] = pop["moismut"]
pop["trimestre"] = (pop["moismut"] - 1) // 3 + 1

print(pop.groupby("trimestre")["mois"].agg(["min", "max"]))

           min  max
trimestre          
1            1    3
2            4    6
3            7    9
4           10   12


### Calculer les neuf tables

Une seule fonction calcule une table, pour une maille et un pas de temps donnés. On l'appelle neuf fois.

Dans la fonction :
1. on regroupe les ventes par zone × période × type de bien, et on calcule le nombre de ventes et le prix médian ;
2. le type de bien passe en colonnes (`unstack`) : on obtient `n_maison`, `n_appart`, `prix_m2_med_maison`, `prix_m2_med_appart` ;
3. une zone sans vente d'un type sur une période a un nombre égal à 0, et un prix vide.

**Contrôle :** dans chaque table, la somme des ventes doit être égale au nombre de ventes après exclusion. Aucune vente ne doit être perdue.

In [ ]:
def table_cible(pop, maille, periode):
    """Prix médian au m² et nombre de ventes par zone × période, maisons et appartements."""
    cles = [maille] + periode
    t = (pop.groupby(cles + ["type_bien"])["prix_m2"]
            .agg(n="count", prix_m2_med="median")
            .unstack("type_bien"))
    # ("n", "maison") devient "n_maison"
    t.columns = [f"{stat}_{typ}" for stat, typ in t.columns]
    t = t.reset_index()
    for col in ["n_maison", "n_appart"]:
        t[col] = t[col].fillna(0).astype(int)
    t[["prix_m2_med_maison", "prix_m2_med_appart"]] = t[["prix_m2_med_maison", "prix_m2_med_appart"]].round(0)
    return t[cles + ["n_appart", "n_maison", "prix_m2_med_appart", "prix_m2_med_maison"]]


# Trois mailles (nom du fichier : colonne de la zone) et trois pas de temps
mailles = {"commune": "code_commune", "bv": "BV2022", "sbv": "sous_bassin_de_vie"}
pas_de_temps = {"mois": ["annee", "mois"], "trimestre": ["annee", "trimestre"], "annee": ["annee"]}

for nom_maille, maille in mailles.items():
    for nom_pas, periode in pas_de_temps.items():
        t = table_cible(pop, maille, periode)
        total = t["n_appart"].sum() + t["n_maison"].sum()
        t.to_parquet(DATA_OUT / f"cible_{nom_maille}_{nom_pas}.parquet", index=False)
        print(f"cible_{nom_maille}_{nom_pas} : {len(t):>9,} lignes | ventes : {total:,} | aucune perdue : {total == len(pop)}")

### Résultat du bloc 4

Neuf tables dans `data/processed/` :

| Fichier | Une ligne = | Lignes |
|---|---|---|
| `cible_commune_mois.parquet` | commune × mois | 2 043 312 |
| `cible_commune_trimestre.parquet` | commune × trimestre | 1 067 694 |
| `cible_commune_annee.parquet` | commune × année | 362 187 |
| `cible_bv_mois.parquet` | bassin de vie × mois | 231 397 |
| `cible_bv_trimestre.parquet` | bassin de vie × trimestre | 77 364 |
| `cible_bv_annee.parquet` | bassin de vie × année | 19 355 |
| `cible_sbv_mois.parquet` | sous-bassin × mois | 260 441 |
| `cible_sbv_trimestre.parquet` | sous-bassin × trimestre | 87 118 |
| `cible_sbv_annee.parquet` | sous-bassin × année | 21 802 |

- Colonnes : la zone (`code_commune`, `BV2022` ou `sous_bassin_de_vie`), la période (`annee`, plus `mois` ou `trimestre`), puis `n_appart`, `n_maison`, `prix_m2_med_appart`, `prix_m2_med_maison`.
- Chaque table compte les 9 655 421 ventes retenues : aucune n'est perdue. Toutes les ventes ont une commune (33 152 communes, arrondissements séparés) un bassin de vie (1 613 bassins, arrondissements dans le bassin de leur ville) et un sous-bassin.
- Les tables couvrent 2014-2025. **L'année 2025 est réservée au test** : elle doit être exclue des analyses exploratoires et de l'entraînement.
- **Aucun seuil de ventes n'est appliqué** : le seuil de 20 ventes par bassin, par année et par type de bien sera appliqué à la modélisation, à partir de `n_maison` et `n_appart`.
- Une zone sans aucune vente sur une période n'a pas de ligne. Au bassin de vie et à l'année, cela n'arrive qu'une fois (19 355 lignes pour 1 613 bassins × 12 années). Une zone peut aussi avoir une ligne sans vente d'un des deux types : son nombre vaut alors 0 et son prix est vide. Les séries mensuelles des petites communes auront des trous.

# Bloc 5 — Filosofi : revenus des ménages par commune

**Contributeurs :** Cécile NAUDET · **Date :** 15/10/2026

Filosofi est une source de l'Insee. Elle décrit le **revenu disponible** des ménages (revenus après impôts et prestations sociales) à partir des données fiscales. On utilise les fichiers par commune, de 2014 à 2021 : un fichier par année, appelée « millésime ».

Les huit fichiers ne sont pas tous au même format :
- 2014 à 2017 : fichiers Excel, dont les noms de colonnes sont à la 6e ligne ;
- 2018 à 2021 : fichiers CSV séparés par des points-virgules ;
- dans tous les cas, les noms de colonnes se terminent par l'année (`Q214` = médiane de 2014) ;
- en 2020 et 2021, les décimales sont écrites avec une virgule (`0,225`) et les valeurs soumises au secret statistique sont notées `s`.

Chaque millésime est publié dans la géographie de l'année suivante (« géographie au 01/01/2015 » pour 2014). Les codes des communes fusionnées d'un millésime à l'autre seront rattachés au bon bassin de vie par la table de liaison du bloc 1.

**Une ligne = une commune × une année.**

### Assembler les huit millésimes

On garde **toutes les colonnes présentes les huit années**, pour pouvoir suivre chaque indicateur dans le temps. Le choix des colonnes utiles se fait plus tard, dans l'EDA et la modélisation.

La cellule procède en quatre temps :
1. lire chaque fichier et retirer l'année à la fin des noms de colonnes (`Q214` devient `Q2`) ;
2. repérer les colonnes communes aux huit années ;
3. garder ces colonnes et empiler les huit tableaux, avec une colonne `annee` ;
4. convertir toutes les colonnes en nombres, sauf le code commune : la virgule est remplacée par un point, et `s` devient une valeur vide.

**Cas particulier de 2014 :** la part des revenus d'activité (`PACT`) n'existe pas encore. On la reconstruit : en 2014, `PTSAC` regroupe ce qui devient `PTSA` + `PCHO` à partir de 2015, donc `PACT = PTSAC + PBEN`.

Les colonnes écartées sont affichées pour chaque année. Cela permet de vérifier qu'aucune colonne n'est perdue à cause d'un nom écrit différemment selon les années.

In [22]:
import re

dossier_filo = DATA_RAW / "Filo_disp_com"


def en_nombre(df, cols):
    """Convertit des colonnes en nombres (virgule -> point, 's' -> vide)."""
    return df[cols].apply(lambda s: pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False),
                                                  errors="coerce"))


# 1. Lire et renommer chaque fichier
fichiers_filo = sorted(dossier_filo.glob("FILO*DISP_COM*"))
print("Fichiers trouvés :", len(fichiers_filo))   # 8 attendus

tableaux = []
for f in fichiers_filo:
    annee = int(re.search(r"20\d\d", f.name).group())
    aa = str(annee)[2:]                       # "14", "15", ...

    if f.suffix in (".xls", ".xlsx"):
        df = pd.read_excel(f, sheet_name="ENSEMBLE", header=5, dtype={"CODGEO": str})
    else:
        df = pd.read_csv(f, sep=";", dtype={"CODGEO": str})

    # On retire l'année à la fin des noms : "Q214" -> "Q2"
    df.columns = [c[:-2] if c.endswith(aa) and c != "CODGEO" else c for c in df.columns]
    df["annee"] = annee

    # 2014 : PACT n'existe pas. PTSAC (2014) = PTSA + PCHO (2015+), donc PACT = PTSAC + PBEN
    if annee == 2014:
        x = en_nombre(df, ["PTSAC", "PBEN"])
        df["PACT"] = x["PTSAC"] + x["PBEN"]

    tableaux.append(df)

# 2. Colonnes présentes les 8 années (intersection)
communes = set(tableaux[0].columns)
for df in tableaux[1:]:
    communes = communes & set(df.columns)
communes = ["CODGEO", "annee"] + sorted(communes - {"CODGEO", "annee"})
print(len(communes), "colonnes communes :", communes)

# Contrôle : ce que chaque année perd
for df in tableaux:
    ecartees = sorted(set(df.columns) - set(communes))
    print(df["annee"].iloc[0], "|", len(ecartees), "écartées :", ecartees[:10])

# 3. Garder les colonnes communes et empiler
filo = pd.concat([df[communes] for df in tableaux], ignore_index=True)

# 4. Tout en nombres, sauf le code commune
cols_num = [c for c in filo.columns if c != "CODGEO"]
filo[cols_num] = en_nombre(filo, cols_num)

print(filo.shape)
filo.head()

Fichiers trouvés : 8


/tmp/ipykernel_45646/2210929812.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["annee"] = annee
/tmp/ipykernel_45646/2210929812.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["annee"] = annee


29 colonnes communes : ['CODGEO', 'annee', 'D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D9', 'GI', 'NBMEN', 'NBPERS', 'NBUC', 'PACT', 'PBEN', 'PIMPOT', 'PPAT', 'PPEN', 'PPFAM', 'PPLOGT', 'PPMINI', 'PPSOC', 'Q1', 'Q2', 'Q3', 'Q3_Q1', 'RD', 'S80S20']
2014 | 2 écartées : ['LIBGEO', 'PTSAC']
2015 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2016 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2017 | 3 écartées : ['LIBGEO', 'PCHO', 'PTSA']
2018 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2019 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2020 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
2021 | 704 écartées : ['AGE1D1', 'AGE1D2', 'AGE1D3', 'AGE1D4', 'AGE1D6', 'AGE1D7', 'AGE1D8', 'AGE1D9', 'AGE1GI', 'AGE1PACT']
(261319, 29)


,CODGEO,annee,D1,D2,D3,D4,D6,D7,D8,D9,...,PPFAM,PPLOGT,PPMINI,PPSOC,Q1,Q2,Q3,Q3_Q1,RD,S80S20
0,01001,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21576.666667,NaN,NaN,NaN,NaN
1,01002,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,21672.857143,NaN,NaN,NaN,NaN
2,01004,2014,10545.666667,13184.666667,15507.333333,17564.0,21840.0,24469.2,27991.333333,33376.5,...,2.8,1.9,1.8,6.5,14340.0,19756.111111,25954.0,11614.0,3.164949,3.792281
3,01005,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,23204.800000,NaN,NaN,NaN,NaN
4,01006,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,22157.500000,NaN,NaN,NaN,NaN


### Contrôler la couverture et les doublons

Trois contrôles :
- **la couverture par année** : on compte les médianes remplies, et pas seulement les lignes. En 2020 et 2021, les communes soumises au secret statistique sont présentes avec des valeurs vides, alors qu'avant 2020 elles étaient absentes ;
- **les doublons** : une commune doit apparaître une seule fois par année ;
- **Paris, Lyon et Marseille** : sont-elles présentes à la fois en ville entière et par arrondissement ?

In [23]:
couverture = filo.groupby("annee").agg(
    lignes=("CODGEO", "size"),
    mediane_remplie=("Q2", "count"),     # count ignore les valeurs vides
    gini_rempli=("GI", "count"),
    d1_rempli=("D1", "count"),
)
print(couverture)

# Une commune doit apparaître une seule fois par année
print("Doublons commune × année :", filo.duplicated(["CODGEO", "annee"]).sum())

# Paris, Lyon, Marseille : commune entière et arrondissements en même temps ?
print(filo.loc[filo["CODGEO"].str.match(r"^(75056|75101|69123|69381|13055|13201)$"), ["CODGEO", "annee"]]
      .groupby("CODGEO").size())

       lignes  mediane_remplie  gini_rempli  d1_rempli
annee                                                 
2014    32974            32974         5222       5222
2015    32252            32252         5257       5257
2016    31789            31789         5279       5279
2017    31745            31745         5304       5304
2018    31330            31330         5307       5307
2019    31361            31361         5336       5336
2020    34939            31336         5351       5351
2021    34929            31325         5346       5346
Doublons commune × année : 0
CODGEO
13055    8
13201    8
69123    8
69381    8
75056    8
75101    8
dtype: int64


### Renommer et enregistrer

Deux colonnes sont renommées pour la jointure avec les autres sources : `CODGEO` devient `code_commune`, et `Q2` (médiane du niveau de vie) devient `niveau_vie_median`.

In [24]:
filo = filo.rename(columns={"CODGEO": "code_commune", "Q2": "niveau_vie_median"})
filo.to_parquet(DATA_OUT / "filosofi_commune_2014_2021.parquet", index=False)
print(filo.shape)

(261319, 29)


### Résultat du bloc 5


- `data/processed/filosofi_commune_2014_2021.parquet` : **261 319 lignes** (commune × année), 29 colonnes, toutes numériques sauf le code commune.
- La médiane du niveau de vie est renseignée pour 31 300 à 33 000 communes selon les années, sur environ 35 000. D1, D9, Gini et les parts de revenus ne sont publiés que pour environ 5 300 communes chaque année (secret statistique).
- Aucun doublon commune × année.


# Bloc 6 — Ménages : structure des ménages par commune

**Contributeurs :** Gabrielle RICHERT **Date :** 16/10/2026

Source : Insee, recensement de la population 2022, base « Couples, familles, ménages » à l'IRIS (géographie au 1er janvier 2024). Un seul millésime : 2022.

L'audit a proposé de résumer les 80 colonnes en **5 ratios** qui décrivent la demande de logement :
- **taux de personnes seules** : besoin de petites surfaces ;
- **part des familles monoparentales** : pression sur les logements moyens et abordables ;
- **part des familles nombreuses** (3 enfants ou plus) : besoin de grands logements ;
- **taille moyenne des ménages** : plus elle est faible, plus il faut de logements pour une même population ;
- **part des couples sans enfant**.

### Deux règles

- **On travaille à la commune, pas à l'IRIS.** DVF+ n'a pas de code IRIS, et le projet n'utilise pas les contours IRIS. On additionne donc les effectifs des IRIS de chaque commune. Paris, Lyon et Marseille sont déjà codées par arrondissement dans la colonne `COM`, comme dans DVF+.
- **On additionne les effectifs, puis on divise ; on ne fait jamais la moyenne des ratios.** Les effectifs bruts sont gardés dans le fichier : ils permettront de recalculer les ratios au bassin de vie de la même façon.

In [25]:
chemin_menages = (DATA_RAW / "base-ic-couples-familles-menages-2022_csv"
                  / "base-ic-couples-familles-menages-2022.CSV")

# Codes lus en texte : garde les zéros du début ("01001") et la Corse ("2A004")
codes_texte = {"IRIS": str, "COM": str, "TYP_IRIS": str, "LAB_IRIS": str}
menages = pd.read_csv(chemin_menages, sep=";", dtype=codes_texte)

# Effectifs utiles pour les 5 ratios
effectifs = ["C22_MEN", "C22_PMEN", "C22_MENPSEUL", "C22_MENFAMMONO",
             "C22_NE24F3", "C22_NE24F4P", "C22_MENCOUPSENF"]

print(menages.shape)
print("Un IRIS = une ligne :", menages["IRIS"].is_unique)
print(menages[effectifs].dtypes)        # doivent être des nombres (float64)
print(menages[["IRIS", "COM"]].head(3))

(49276, 80)
Un IRIS = une ligne : True
C22_MEN            float64
C22_PMEN           float64
C22_MENPSEUL       float64
C22_MENFAMMONO     float64
C22_NE24F3         float64
C22_NE24F4P        float64
C22_MENCOUPSENF    float64
dtype: object
        IRIS    COM
0  010010000  01001
1  010020000  01002
2  010040101  01004


### Additionner les IRIS par commune

Une commune non découpée correspond à un seul IRIS ; une grande commune en compte plusieurs. On additionne les effectifs de tous les IRIS d'une même commune (`groupby("COM").sum()`).

In [26]:
menages_com = menages.groupby("COM")[effectifs].sum().reset_index()

print("Communes (et arrondissements) :", len(menages_com))
print("Une commune = une ligne :", menages_com["COM"].is_unique)
menages_com.head()

Communes (et arrondissements) : 34960
Une commune = une ligne : True


,COM,C22_MEN,C22_PMEN,C22_MENPSEUL,C22_MENFAMMONO,C22_NE24F3,C22_NE24F4P,C22_MENCOUPSENF
0,01001,353.417143,859.000000,73.628571,24.542857,14.725714,4.908571,117.805714
1,01002,119.127273,273.000000,39.709091,4.963636,4.963636,0.000000,34.745455
2,01004,7064.010268,15061.008969,2887.872092,683.510217,394.937953,126.569421,1725.738986
3,01005,819.836882,1917.000000,218.500246,39.197152,34.302039,4.676972,247.996644
4,01006,57.000000,114.000000,15.545455,10.363636,0.000000,0.000000,20.727273


### Calculer les 5 ratios

Chaque ratio est un effectif divisé par le nombre de ménages de la commune (`C22_MEN`).

Une commune sans aucun ménage (zone d'activité, commune non habitée) n'a pas de ratio : on laisse la valeur vide. Mettre 0 serait faux, car « 0 % de personnes seules » est une vraie valeur, qui fausserait les moyennes.

Pour cela, on remplace d'abord les communes à 0 ménage par une valeur vide dans le dénominateur (`where`) : une division par une valeur vide donne une valeur vide.

In [27]:
# Nombre de ménages, vide si la commune n'en a aucun
men = menages_com["C22_MEN"].where(menages_com["C22_MEN"] > 0)

menages_com["taux_personnes_seules"] = menages_com["C22_MENPSEUL"] / men
menages_com["part_familles_monoparentales"] = menages_com["C22_MENFAMMONO"] / men
menages_com["part_familles_nombreuses"] = (menages_com["C22_NE24F3"] + menages_com["C22_NE24F4P"]) / men
menages_com["taille_moyenne_menage"] = menages_com["C22_PMEN"] / men
menages_com["part_couples_sans_enfants"] = menages_com["C22_MENCOUPSENF"] / men

ratios = ["taux_personnes_seules", "part_familles_monoparentales", "part_familles_nombreuses",
          "taille_moyenne_menage", "part_couples_sans_enfants"]

print("Communes sans ménage (ratios vides) :", men.isna().sum())
menages_com[ratios].describe().round(3)

Communes sans ménage (ratios vides) : 11


,taux_personnes_seules,part_familles_monoparentales,part_familles_nombreuses,taille_moyenne_menage,part_couples_sans_enfants
count,34949.000,34949.000,34949.000,34949.000,34949.000
mean,0.311,0.078,0.046,2.239,0.325
std,0.091,0.058,0.038,0.261,0.083
min,0.000,0.000,0.000,1.000,0.000
25%,0.250,0.043,0.022,2.071,0.279
50%,0.301,0.076,0.044,2.250,0.325
75%,0.364,0.106,0.065,2.405,0.369
max,1.000,1.000,0.572,5.721,1.000


### Enregistrer

Le fichier garde le code commune, les 7 effectifs bruts et les 5 ratios. Le code commune est renommé `code_commune`, comme dans les autres tables. On ajoute l'année du recensement (`annee_rp`) : la même valeur, 2022, servira pour toutes les années de vente.

In [28]:
menages_com = menages_com.rename(columns={"COM": "code_commune"})
menages_com["annee_rp"] = 2022

menages_com.to_parquet(DATA_OUT / "menages_commune_2022.parquet", index=False)
print(menages_com.shape)

(34960, 14)


### Partie 2 : effectifs des ménages par commune, 2019 à 2022

**Contributeurs :** Gabrielle RICHERT & Christian · **Date :** 27/10/2026, mis à jour le 03/11/2026

Source : Insee, recensements de la population 2019 à 2022, base « Couples, familles, ménages » à l'IRIS.

On garde seulement les **effectifs** par commune et par année : ils s'additionnent à n'importe quelle maille. Le rattachement aux sous-bassins, les 5 ratios et le prolongement à 2023 et 2024 sont faits au feature engineering (section Ménages), comme pour les autres sources.

#### Lire les 4 millésimes et additionner les IRIS par commune

In [ ]:
ANNEES_RP = [2019, 2020, 2021, 2022]
dossier_menages = DATA_RAW / "base-ic-couples-familles-menages-2022_csv"

# Effectifs utiles aux 5 ratios (noms sans l'année : C_MEN pour C19_MEN, C20_MEN...)
EFFECTIFS_MENAGES = ["C_MEN", "C_MENPSEUL", "C_PMEN", "C_FAM",
                     "C_FAMMONO", "C_COUPSENF", "C_NE24F3", "C_NE24F4P"]

morceaux = []
for an in ANNEES_RP:
    df = pd.read_csv(dossier_menages / f"base-ic-couples-familles-menages-{an}.CSV",
                     sep=";", dtype={"IRIS": str, "COM": str}, low_memory=False)
    # Année retirée des noms de colonnes : C19_MEN -> C_MEN
    df.columns = [re.sub(rf"^([CP]){str(an)[2:]}_", r"\1_", c) for c in df.columns]
    # Somme des IRIS par commune ; min_count=1 : une somme de valeurs vides reste vide
    com_an = df.groupby("COM", as_index=False)[EFFECTIFS_MENAGES].sum(min_count=1)
    com_an["annee"] = an
    morceaux.append(com_an)
    print(f"{an} : {len(df)} IRIS -> {len(com_an)} communes")

menages_com_an = pd.concat(morceaux, ignore_index=True)
print("Une commune x année = une ligne :", not menages_com_an.duplicated(["COM", "annee"]).any())

#### Contrôler et enregistrer

Une ligne par commune et par année, avec les 8 effectifs. Le code commune est renommé `code_commune`, comme dans les autres tables. Les codes sont ceux de chaque millésime : le feature engineering les rattache aux sous-bassins par la table de liaison (`CODGEO_INI`), qui connaît les anciens codes.

In [ ]:
menages_com_an = menages_com_an.rename(columns={"COM": "code_commune"})
menages_com_an = menages_com_an[["code_commune", "annee"] + EFFECTIFS_MENAGES]
print("Bloc 6 (partie 2) - lignes :", len(menages_com_an),
      "| années :", sorted(menages_com_an["annee"].unique()),
      "| une commune x année = une ligne :", not menages_com_an.duplicated(["code_commune", "annee"]).any(),
      "| effectifs vides :", menages_com_an[EFFECTIFS_MENAGES].isna().sum().sum())
menages_com_an.to_parquet(DATA_OUT / "menages_commune_2019_2022.parquet", index=False)

### Résultat du bloc 6

**Partie 1 : par commune, recensement 2022**
- `data/processed/menages_commune_2022.parquet` : **34 960 communes et arrondissements**, 14 colonnes (code commune, 7 effectifs bruts, 5 ratios, année du recensement).
- 11 communes sans ménage : leurs ratios sont vides.
- Ratios exprimés entre 0 et 1 (sauf la taille moyenne des ménages, en nombre de personnes), tous rapportés au nombre de ménages.

**Partie 2 : par commune et par année, recensements 2019 à 2022**
- `data/processed/menages_commune_2019_2022.parquet` : une ligne par commune et par année (codes de chaque millésime), les 8 effectifs des ménages et des familles.
- Ratios, rattachement aux sous-bassins et prolongement à 2023 et 2024 : feature engineering, section Ménages.

# Bloc 7 — Parc de logement par commune

**Contributeurs :** Théophyle CHEVET · **Date :** 20/10/2026

Source : Insee, recensement de la population 2022, base « Logement » à l'IRIS (géographie au 1er janvier 2024). Un seul millésime : 2022.

### Ce que l'exploration a décidé

- **Colonnes inutiles retirées** : 10 variables propres à l'outre-mer (vides en métropole), 5 de chauffage (vides en outre-mer), `LAB_IRIS` (non documenté) et `TYP_IRIS` (sans objet une fois à la commune).
- **IRIS sans logement retirés** (152 zones d'activité ou parcs) : tous leurs effectifs sont nuls.
- **Agrégation par commune** : on additionne les effectifs des IRIS, puis on calcule les indicateurs. Paris, Lyon et Marseille restent par arrondissement, comme dans DVF+.
- **19 indicateurs**, en pourcentage (0 à 100) ou en moyenne.
- **Les effectifs bruts sont conservés** pour recalculer les indicateurs au bassin de vie.

In [37]:
chemin_logement = DATA_RAW / "base-ic-logement-2022_csv" / "base-ic-logement-2022.CSV"

# Codes lus en texte : garde les zéros du début ("01001") et la Corse ("2A004")
codes_texte = {"IRIS": str, "COM": str, "TYP_IRIS": str, "LAB_IRIS": str}
logement = pd.read_csv(chemin_logement, sep=";", dtype=codes_texte)
print("Chargé :", logement.shape)

# 10 variables propres à l'outre-mer (vides en métropole) et 5 de chauffage (vides en outre-mer)
colonnes_dom = ["P22_RP_ELEC", "P22_RP_EAUCH", "P22_RP_BDWC", "P22_RP_CHOS", "P22_RP_CLIM",
                "P22_RP_TTEGOU", "P22_RP_HABFOR", "P22_RP_CASE", "P22_RP_MIBOIS", "P22_RP_MIDUR"]
colonnes_chauffage = ["P22_RP_CGAZV", "P22_RP_CFIOUL", "P22_RP_CELEC", "P22_RP_CGAZB", "P22_RP_CAUT"]
# LAB_IRIS : label de qualité non documenté ; TYP_IRIS n'a plus de sens après passage à la commune
logement = logement.drop(columns=colonnes_dom + colonnes_chauffage + ["LAB_IRIS", "TYP_IRIS"])

# IRIS sans logement (zones d'activité, parcs...) : tous les effectifs à 0
avant = len(logement)
logement = logement[logement["P22_LOG"] > 0]

print("Après nettoyage :", logement.shape, "| IRIS sans logement supprimés :", avant - len(logement))

Chargé : (49276, 101)
Après nettoyage : (49124, 84) | IRIS sans logement supprimés : 152


### Additionner les IRIS par commune

On additionne tous les effectifs des IRIS d'une même commune ou d'un même arrondissement. Les variables « totaux » (nombre de pièces, de personnes, années d'ancienneté) s'additionnent aussi : on les divisera ensuite pour obtenir des moyennes.

On compte aussi le nombre d'IRIS par commune (`nb_iris`) : 1 pour une commune non découpée.

In [38]:
# Toutes les colonnes d'effectifs (celles qui commencent par P22_ ou C22_)
colonnes_effectifs = [c for c in logement.columns if c.startswith("P22_") or c.startswith("C22_")]

# On ADDITIONNE les effectifs des IRIS d'une même commune (jamais de moyenne de taux)
commune = logement.groupby("COM")[colonnes_effectifs].sum()

# Nombre d'IRIS par commune (1 = commune non découpée)
commune["nb_iris"] = logement.groupby("COM")["IRIS"].count()
commune = commune.reset_index()

print("Communes et arrondissements :", len(commune))
print("Une commune = une ligne :", commune["COM"].is_unique)

Communes et arrondissements : 34955
Une commune = une ligne : True


### Créer les 19 indicateurs 

Les effectifs bruts dépendent de la taille de la commune : on les transforme en parts (%) ou en moyennes, comparables d'une commune à l'autre.

Trois dénominateurs selon l'indicateur :
- **tous les logements** (`P22_LOG`) : vacance, résidences secondaires, type d'habitat ;
- **les résidences principales** (`P22_RP`) : taille, âge du parc, statut d'occupation, suroccupation ;
- **les ménages** (`P22_MEN`) : mobilité, voiture, stationnement.

La fonction `ratio` protège contre la division par 0 : une commune dont le dénominateur vaut 0 reçoit une valeur vide, et non une erreur ou l'infini.

In [39]:
def ratio(numerateur, denominateur):
    """Part en % ; vide si le dénominateur vaut 0."""
    return (numerateur / denominateur.replace(0, np.nan)) * 100

c = commune          # nom court
log_ = c["P22_LOG"]  # dénominateur : tous les logements
rp = c["P22_RP"]     # dénominateur : résidences principales
men = c["P22_MEN"]   # dénominateur : ménages

# Marché : vacance et tourisme
c["taux_vacance"] = ratio(c["P22_LOGVAC"], log_)
c["part_rsec"] = ratio(c["P22_RSECOCC"], log_)

# Type d'habitat
c["part_maisons"] = ratio(c["P22_MAISON"], log_)
c["part_apparts"] = ratio(c["P22_APPART"], log_)

# Taille des logements (résidences principales)
c["part_moins40m2"] = ratio(c["P22_RP_M30M2"] + c["P22_RP_3040M2"], rp)
c["part_100m2_plus"] = ratio(c["P22_RP_100120M2"] + c["P22_RP_120M2P"], rp)
c["nb_pieces_moyen"] = c["P22_NBPI_RP"] / rp.replace(0, np.nan)          # moyenne, pas un %

# Âge du parc
c["part_avant1946"] = ratio(c["P22_RP_ACH1919"] + c["P22_RP_ACH1945"], rp)
c["part_2006_2019"] = ratio(c["P22_RP_ACH2019"], rp)

# Profil social / statut d'occupation
c["part_proprio"] = ratio(c["P22_RP_PROP"], rp)
c["part_locataires"] = ratio(c["P22_RP_LOC"], rp)
c["part_hlm"] = ratio(c["P22_RP_LOCHLMV"], rp)

# Mobilité résidentielle
c["part_emmenages_2ans"] = ratio(c["P22_MEN_ANEM0002"], men)
c["anciennete_moyenne"] = c["P22_ANEM_RP"] / rp.replace(0, np.nan)      # en années

# Ménages, voiture, stationnement, suroccupation
c["taille_menage"] = c["P22_NPER_RP"] / rp.replace(0, np.nan)           # personnes par ménage
c["part_sans_voiture"] = 100 - ratio(c["P22_RP_VOIT1P"], men)
c["part_2voitures"] = ratio(c["P22_RP_VOIT2P"], men)
c["part_parking"] = ratio(c["P22_RP_GARL"], men)
c["part_suroccupes"] = ratio(c["C22_RP_SUROCC_MOD"] + c["C22_RP_SUROCC_ACC"], rp)

indicateurs = ["taux_vacance", "part_rsec", "part_maisons", "part_apparts",
               "part_moins40m2", "part_100m2_plus", "nb_pieces_moyen",
               "part_avant1946", "part_2006_2019",
               "part_proprio", "part_locataires", "part_hlm",
               "part_emmenages_2ans", "anciennete_moyenne",
               "taille_menage", "part_sans_voiture", "part_2voitures", "part_parking", "part_suroccupes"]

print("Valeurs manquantes par indicateur :")
print(c[indicateurs].isna().sum())

c = c.copy()   # réorganise le tableau en mémoire après l'ajout des 19 colonnes

Valeurs manquantes par indicateur :
taux_vacance           0
part_rsec              0
part_maisons           0
part_apparts           0
part_moins40m2         1
part_100m2_plus        1
nb_pieces_moyen        1
part_avant1946         1
part_2006_2019         1
part_proprio           1
part_locataires        1
part_hlm               1
part_emmenages_2ans    1
anciennete_moyenne     1
taille_menage          1
part_sans_voiture      1
part_2voitures         1
part_parking           1
part_suroccupes        1
dtype: int64


/tmp/ipykernel_45646/3065487624.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  c["part_parking"] = ratio(c["P22_RP_GARL"], men)
/tmp/ipykernel_45646/3065487624.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  c["part_suroccupes"] = ratio(c["C22_RP_SUROCC_MOD"] + c["C22_RP_SUROCC_ACC"], rp)


### Colonnes d'aide et contrôles 

On ajoute cinq colonnes utiles pour la suite :
- `dep` : le département ;
- `alsace_moselle` : ces communes sont absentes de DVF+ ; on les signale au lieu de les supprimer, le choix se fera à la jointure ;
- `petite_commune` : moins de 50 logements, des indicateurs fragiles ;
- `log_nb_logements` : la taille de la commune en logarithme, car sa distribution est très asymétrique ;
- `annee_rp` : le millésime du recensement (2022).

Puis on contrôle que chaque part reste entre 0 et 100.

In [40]:
# Département (2 caractères en métropole, 3 en outre-mer)
c["dep"] = np.where(c["COM"].str[:2] == "97", c["COM"].str[:3], c["COM"].str[:2])

# Alsace-Moselle : absente de DVF+ -> on la SIGNALE au lieu de la supprimer
c["alsace_moselle"] = c["dep"].isin(["57", "67", "68"])

# Fiabilité : en dessous de 50 logements, les taux estimés sont fragiles
c["petite_commune"] = c["P22_LOG"] < 50

# Taille de la commune, en logarithme (distribution très asymétrique)
c["log_nb_logements"] = np.log10(c["P22_LOG"])

# Millésime du recensement
c["annee_rp"] = 2022

# Contrôle : les parts doivent être comprises entre 0 et 100
for col in indicateurs:
    if col.startswith("part_") or col == "taux_vacance":
        hors_bornes = ((c[col] < -0.01) | (c[col] > 100.01)).sum()
        if hors_bornes > 0:
            print("Attention :", col, "a", hors_bornes, "valeurs hors de [0, 100]")

print("Communes signalées Alsace-Moselle :", c["alsace_moselle"].sum(),
      "| petites communes (< 50 logements) :", c["petite_commune"].sum())
c[indicateurs].describe().round(1).T

Communes signalées Alsace-Moselle : 1605 | petites communes (< 50 logements) : 2022


,count,mean,std,min,25%,50%,75%,max
taux_vacance,34955.0,8.5,4.8,0.0,5.2,7.7,11.0,100.0
part_rsec,34955.0,13.1,15.2,0.0,2.7,7.2,17.6,97.2
part_maisons,34955.0,89.6,14.8,0.3,87.9,95.1,98.1,100.0
part_apparts,34955.0,9.6,14.7,0.0,1.3,4.0,11.0,99.1
part_moins40m2,34954.0,2.9,3.8,0.0,0.7,1.8,3.6,69.3
part_100m2_plus,34954.0,51.5,14.3,0.0,42.8,52.2,61.3,100.0
nb_pieces_moyen,34954.0,4.8,0.5,2.0,4.6,4.8,5.1,7.0
part_avant1946,34954.0,34.0,17.4,0.0,20.7,32.0,45.5,100.0
part_2006_2019,34954.0,14.5,8.0,0.0,8.7,13.5,19.2,68.3
part_proprio,34954.0,78.6,11.0,0.0,73.8,80.9,86.0,100.0


### Enregistrer 

On garde les identifiants, les effectifs bruts (numérateurs et dénominateurs de tous les indicateurs, pour pouvoir les recalculer au bassin de vie), les 19 indicateurs et les colonnes d'aide. Le code commune est renommé `code_commune`, comme dans les autres tables.

In [41]:
effectifs_utiles = ["P22_LOG", "P22_RP", "P22_MEN",
                    "P22_LOGVAC", "P22_RSECOCC", "P22_MAISON", "P22_APPART",
                    "P22_RP_M30M2", "P22_RP_3040M2", "P22_RP_100120M2", "P22_RP_120M2P", "P22_NBPI_RP",
                    "P22_RP_ACH1919", "P22_RP_ACH1945", "P22_RP_ACH2019",
                    "P22_RP_PROP", "P22_RP_LOC", "P22_RP_LOCHLMV",
                    "P22_MEN_ANEM0002", "P22_ANEM_RP", "P22_NPER_RP",
                    "P22_RP_VOIT1P", "P22_RP_VOIT2P", "P22_RP_GARL",
                    "C22_RP_SUROCC_MOD", "C22_RP_SUROCC_ACC"]

colonnes_finales = (["COM", "dep", "annee_rp", "nb_iris", "log_nb_logements"]
                    + effectifs_utiles + indicateurs + ["alsace_moselle", "petite_commune"])

resultat = c[colonnes_finales].rename(columns={"COM": "code_commune"})
resultat.to_parquet(DATA_OUT / "logement_commune_2022.parquet", index=False)
print("Enregistré :", resultat.shape)

Enregistré : (34955, 52)


### Résultat du bloc 7

- `data/processed/logement_commune_2022.parquet` : **34 955 communes et arrondissements**, 52 colonnes (identifiants, 26 effectifs bruts, 19 indicateurs, colonnes d'aide).
- Indicateurs en pourcentage (0 à 100), sauf le nombre moyen de pièces, l'ancienneté moyenne d'emménagement (en années) et la taille moyenne des ménages (en personnes).
- Une commune sans résidence principale : ses indicateurs calculés sur les résidences principales ou les ménages sont vides.



# Bloc 8 — Délinquance enregistrée par commune

**Contributeurs :** Cécile NAUDET · **Date :** 17/10/2026 (audit), 20/10/2026

Source : base communale du SSMSI, 2016 à 2025, géographie au 1er janvier 2026. Une ligne = une commune × une année × un indicateur (15 indicateurs).

### Ce que l'audit a décidé

- **Paris, Lyon et Marseille** sont présentes deux fois : ville entière et arrondissements, qui décrivent exactement les mêmes faits. On garde **les arrondissements**, comme dans DVF+ et les tables cibles, et on retire la ville entière.
- **Les communes masquées par le secret statistique** n'ont pas de nombre. On le remplace par le complément fourni par la base (`complement_info_nombre`, la moyenne des communes masquées du département), et non par 0 : une valeur masquée signifie « quelques faits », pas « aucun ».
- **Format large** : une ligne par commune × année, une colonne par indicateur.
- **Les effectifs sont conservés** (nombre de faits, population, logements) pour recalculer les taux au bassin de vie en additionnant, jamais en faisant la moyenne des taux communaux.

Le choix des indicateurs à garder se fera dans l'EDA et la modélisation : on les prépare tous ici.

In [42]:
import re
import unicodedata

delinq = pd.read_parquet(DATA_RAW / "dataset_delinquance_communale.parquet")

print(delinq.shape)
print("Codes :", delinq["CODGEO_2026"].nunique(), "| années :", sorted(delinq["annee"].unique()))
print(delinq["indicateur"].value_counts())

(5238000, 13)
Codes : 34920 | années : [np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
indicateur
Violences physiques intrafamiliales               349200
Violences physiques hors cadre familial           349200
Violences sexuelles                               349200
Vols avec armes                                   349200
Vols violents sans arme                           349200
Vols sans violence contre des personnes           349200
Cambriolages de logement                          349200
Vols de véhicule                                  349200
Vols dans les véhicules                           349200
Vols d'accessoires sur véhicules                  349200
Destructions et dégradations volontaires          349200
Usage de stupéfiants                              349200
Usage de stupéfiants (AFD)                        349200
Trafic de stupéfiants                           

### Retirer la ville entière de Paris, Lyon et Marseille

On garde les arrondissements et on retire les trois codes de ville entière (75056, 69123, 13055). Les arrondissements se rattachent au bassin de vie de leur ville grâce à la table de liaison du bloc 1.

In [43]:
villes_entieres = ["75056", "69123", "13055"]
delinq = delinq[~delinq["CODGEO_2026"].isin(villes_entieres)]

print("Codes restants :", delinq["CODGEO_2026"].nunique())
print("Lignes :", len(delinq))

Codes restants : 34917
Lignes : 5237550


### Remplacer les nombres masqués par le complément départemental

Pour une ligne masquée par le secret statistique (`est_diffuse` = « ndiff »), le nombre est vide, mais la base fournit `complement_info_nombre` : le nombre moyen de faits des communes masquées du même département, la même année, pour le même indicateur.

On crée une colonne `nombre_estime` : le nombre publié quand il existe, sinon le complément (`fillna`). L'audit a montré que chaque ligne a soit l'un, soit l'autre : il ne doit plus rester de valeur vide.

In [44]:
delinq["nombre_estime"] = delinq["nombre"].fillna(delinq["complement_info_nombre"])

print("Lignes masquées (nombre estimé) :", (delinq["est_diffuse"] == "ndiff").sum())
print("Nombres encore vides :", delinq["nombre_estime"].isna().sum())

Lignes masquées (nombre estimé) : 2425872
Nombres encore vides : 0


### Passer en format large

On passe d'une ligne par commune × année × indicateur à **une ligne par commune × année**, avec une colonne par indicateur (`pivot`).

Les noms des indicateurs contiennent des espaces, des accents et des apostrophes, peu pratiques comme noms de colonnes. Une petite fonction les transforme : « Vols d'accessoires sur véhicules » devient `vols_d_accessoires_sur_vehicules`. On ajoute le préfixe `nb_` (nombre de faits).

On ajoute ensuite :
- la population et le nombre de logements de la commune (`insee_pop`, `insee_log`), identiques pour les 15 indicateurs d'une même commune ;
- `n_indicateurs_estimes` : le nombre d'indicateurs masqués, donc estimés par le complément, pour la commune cette année-là. Il permet de repérer les communes où les chiffres sont surtout estimés.

In [45]:
def nom_court(libelle):
    """'Vols d'accessoires sur véhicules' -> 'vols_d_accessoires_sur_vehicules'"""
    # On retire les accents, puis on remplace tout ce qui n'est pas lettre ou chiffre par "_"
    sans_accent = unicodedata.normalize("NFKD", libelle).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sans_accent.lower()).strip("_")


# Une colonne par indicateur
large = delinq.pivot(index=["CODGEO_2026", "annee"], columns="indicateur", values="nombre_estime")
large.columns = ["nb_" + nom_court(c) for c in large.columns]

# Population et logements (mêmes valeurs pour tous les indicateurs d'une commune)
denominateurs = delinq.groupby(["CODGEO_2026", "annee"])[["insee_pop", "insee_log"]].first()

# Nombre d'indicateurs masqués par commune et par année
masques = ((delinq["est_diffuse"] == "ndiff")
           .groupby([delinq["CODGEO_2026"], delinq["annee"]]).sum()
           .rename("n_indicateurs_estimes"))

large = large.join(denominateurs).join(masques).reset_index()

print(large.shape)
print(large.columns.tolist())

(349170, 20)
['CODGEO_2026', 'annee', 'nb_violences_physiques_intrafamiliales', 'nb_violences_physiques_hors_cadre_familial', 'nb_violences_sexuelles', 'nb_vols_avec_armes', 'nb_vols_violents_sans_arme', 'nb_vols_sans_violence_contre_des_personnes', 'nb_cambriolages_de_logement', 'nb_vols_de_vehicule', 'nb_vols_dans_les_vehicules', 'nb_vols_d_accessoires_sur_vehicules', 'nb_destructions_et_degradations_volontaires', 'nb_usage_de_stupefiants', 'nb_usage_de_stupefiants_afd', 'nb_trafic_de_stupefiants', 'nb_escroqueries_et_fraudes_aux_moyens_de_paiement', 'insee_pop', 'insee_log', 'n_indicateurs_estimes']


### Calculer les taux pour 1 000 habitants

Pour chaque indicateur : nombre de faits ÷ population × 1 000. Seule exception, comme dans la base du SSMSI : les **cambriolages de logement** se rapportent à 1 000 **logements**.

Une commune sans habitant (les six villages détruits de la Meuse) n'a pas de taux : on laisse la valeur vide.

Ces taux communaux sont **instables dans les petites communes** (l'audit l'a montré) : ils servent pour les modèles à la maille communale. Au bassin de vie, il faudra les recalculer à partir des nombres et des populations.

In [46]:
# Dénominateurs vides si la commune n'a ni habitant ni logement
pop = large["insee_pop"].where(large["insee_pop"] > 0)
log = large["insee_log"].where(large["insee_log"] > 0)

colonnes_nb = [c for c in large.columns if c.startswith("nb_")]
for col in colonnes_nb:
    denominateur = log if col == "nb_cambriolages_de_logement" else pop
    large["taux_" + col[3:]] = large[col] / denominateur * 1000

large.filter(like="taux_").describe().round(2)

,taux_violences_physiques_intrafamiliales,taux_violences_physiques_hors_cadre_familial,taux_violences_sexuelles,taux_vols_avec_armes,taux_vols_violents_sans_arme,taux_vols_sans_violence_contre_des_personnes,taux_cambriolages_de_logement,taux_vols_de_vehicule,taux_vols_dans_les_vehicules,taux_vols_d_accessoires_sur_vehicules,taux_destructions_et_degradations_volontaires,taux_usage_de_stupefiants,taux_usage_de_stupefiants_afd,taux_trafic_de_stupefiants,taux_escroqueries_et_fraudes_aux_moyens_de_paiement
count,349110.00,349110.00,349110.00,349110.00,349110.00,349110.00,349120.00,349110.00,349110.00,349110.00,349110.00,349110.00,349110.00,349110.00,349110.00
mean,3.08,1.91,1.69,0.05,0.16,4.64,7.43,1.80,2.18,1.56,7.14,2.07,0.61,0.41,5.95
std,4.79,4.41,3.38,0.35,0.77,11.56,9.06,3.53,5.79,4.60,13.25,6.94,4.99,1.99,7.46
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,1.16,1.83,0.00,0.00,0.00,2.77,0.00,0.00,0.00,2.12
50%,1.82,0.76,0.60,0.00,0.00,2.92,5.14,0.76,0.84,0.36,5.24,0.46,0.00,0.00,4.23
75%,4.01,2.46,2.09,0.00,0.00,5.72,9.91,2.33,2.63,1.82,8.93,2.36,0.00,0.00,7.33
max,262.88,898.44,214.46,47.06,50.20,1859.50,360.96,316.73,608.70,736.43,2689.81,902.70,826.71,356.67,641.53


### Enregistrer

Le code commune est renommé `code_commune`, comme dans les autres tables.

In [47]:
large = large.rename(columns={"CODGEO_2026": "code_commune"})

print("Une commune × année = une ligne :", not large.duplicated(["code_commune", "annee"]).any())
large.to_parquet(DATA_OUT / "delinquance_commune_annee.parquet", index=False)
print(large.shape)

Une commune × année = une ligne : True
(349170, 35)


### Résultat du bloc 8

- `data/processed/delinquance_commune_annee.parquet` : **349 170 lignes** (34 917 communes et arrondissements × 10 années, 2016-2025), 35 colonnes :
  - 15 nombres de faits (`nb_...`), dont les valeurs masquées remplacées par le complément départemental ;
  - 15 taux pour 1 000 habitants (`taux_...`), pour 1 000 logements pour les cambriolages ;
  - la population, les logements et le nombre d'indicateurs estimés (`n_indicateurs_estimes`).
- Paris, Lyon et Marseille par arrondissement ; la ville entière est retirée.
- Taux vides pour 6 communes sans habitant (Meuse).
- Avec le remplacement des valeurs masquées, les taux médians ne sont plus nuls : les communes masquées reçoivent la moyenne départementale des communes masquées.



# Bloc 9 — LOVAC : logements privés vacants par commune

**Contributeurs :** Théophile CHEVET · **Date :** 20/10/2026

Source : LOVAC (Cerema, à partir des fichiers fiscaux), version open data, millésimes 2020 à 2026, géographie au 1er janvier 2026. Une ligne par commune ; les millésimes sont en colonnes. Le **millésime N décrit la situation au 1er janvier N-1**.

### Ce que l'exploration a décidé

- **Secret statistique** : les valeurs cachées sont notées « s ». On les remplace par une valeur vide (pas par 0), puis on convertit en nombres.
- **Format long** : une ligne par commune × millésime, pour pouvoir joindre avec les ventes de chaque année.
- **Dates** : on ajoute l'année de situation (N-1) et l'année de vente associée (N).
- **3 indicateurs** : taux de vacance du parc privé, taux de vacance de plus de 2 ans, part de la vacance longue parmi les vacants.
- **Taux cachés** : remplacés par le taux de l'intercommunalité (EPCI), avec une colonne qui signale ces valeurs.
- **Signalements** : rupture de série (millésimes 2024 et suivants), Alsace-Moselle, Mayotte.

In [48]:
chemin_lovac = DATA_RAW / "lovac-open-data-2020-a-2026.xlsx"

# Feuille communale, tout en texte : les "s" empêchent de lire les chiffres en nombres
lovac = pd.read_excel(chemin_lovac, sheet_name="COM open data", dtype=str)
print("Chargé :", lovac.shape)

# Colonnes de chiffres : pp_ (parc privé) et ff_ (fichiers fonciers)
colonnes_chiffres = [c for c in lovac.columns if c.startswith("pp_") or c.startswith("ff_")]

# "s" = valeur cachée (secret statistique), pas zéro : on la remplace par une valeur vide
for col in colonnes_chiffres:
    lovac[col] = pd.to_numeric(lovac[col].replace("s", np.nan))

print("Valeurs manquantes (secret statistique) :", lovac[colonnes_chiffres].isna().sum().sum())

Chargé : (34912, 28)
Valeurs manquantes (secret statistique) : 310532


### Passer en format long et ajouter les dates 

Dans le fichier, chaque millésime a ses propres colonnes (`pp_vacant_20`, `pp_vacant_21`…). On empile les 7 millésimes pour obtenir une ligne par commune et par millésime, avec trois effectifs : les logements vacants, les vacants depuis plus de 2 ans et le parc privé total. Le parc privé n'existe pas pour le millésime 2026 : il reste vide.

Deux colonnes de dates :
- `annee_situation` = millésime − 1 : la date réelle de la situation décrite ;
- `annee_vente_associee` = millésime : pour une vente de l'année N, on utilise le millésime N, qui décrit la situation au 1er janvier N-1.

In [49]:
morceaux = []
for annee in range(2020, 2027):
    suffixe = str(annee)[2:]                              # 2023 -> "23"
    morceau = pd.DataFrame({
        "COM": lovac["CODGEO_26"],
        "EPCI": lovac["EPCI_26"],                         # intercommunalité, pour compléter les taux cachés
        "millesime": annee,
        "vacants": lovac["pp_vacant_" + suffixe],
        "vacants_2ans": lovac["pp_vacant_plus_2ans_" + suffixe],
    })
    # Le parc privé total n'existe pas pour le millésime 2026
    if "ff_pp_total_" + suffixe in lovac.columns:
        morceau["parc_prive"] = lovac["ff_pp_total_" + suffixe]
    else:
        morceau["parc_prive"] = np.nan
    morceaux.append(morceau)

lovac_long = pd.concat(morceaux, ignore_index=True)

# Le millésime N décrit la situation au 1er janvier N-1
lovac_long["annee_situation"] = lovac_long["millesime"] - 1
# Pour une vente de l'année N, on utilise le millésime N
lovac_long["annee_vente_associee"] = lovac_long["millesime"]

print("Format long :", lovac_long.shape)

Format long : (244384, 8)


### Créer les indicateurs et les signalements 

Trois indicateurs en pourcentage :
- **taux de vacance** du parc privé : vacants ÷ parc privé ;
- **taux de vacance de plus de 2 ans** (vacance « structurelle ») : vacants depuis plus de 2 ans ÷ parc privé ;
- **part de la vacance longue parmi les vacants** : vacants depuis plus de 2 ans ÷ vacants. Elle ne dépend pas du parc total, souvent caché.

Trois signalements :
- `apres_rupture` : millésimes 2024 et suivants. La fin de la taxe d'habitation a changé la collecte, et ces millésimes ne se comparent pas directement aux précédents ;
- `alsace_moselle` : absente de DVF+ ;
- `mayotte` : absente de DVF.

In [50]:
def ratio(numerateur, denominateur):
    """Part en % ; vide si le dénominateur vaut 0."""
    return (numerateur / denominateur.replace(0, np.nan)) * 100

# Indicateurs
lovac_long["taux_vacance_lovac"] = ratio(lovac_long["vacants"], lovac_long["parc_prive"])
lovac_long["taux_vacance_2ans"] = ratio(lovac_long["vacants_2ans"], lovac_long["parc_prive"])
lovac_long["part_longue_parmi_vacants"] = ratio(lovac_long["vacants_2ans"], lovac_long["vacants"])

# Rupture de série : millésimes 2024 et suivants
lovac_long["apres_rupture"] = lovac_long["millesime"] >= 2024

# Département (2 caractères en métropole, 3 en outre-mer), Alsace-Moselle, Mayotte
lovac_long["dep"] = np.where(lovac_long["COM"].str[:2] == "97",
                             lovac_long["COM"].str[:3], lovac_long["COM"].str[:2])
lovac_long["alsace_moselle"] = lovac_long["dep"].isin(["57", "67", "68"])
lovac_long["mayotte"] = lovac_long["dep"] == "976"

lovac_long[["taux_vacance_lovac", "taux_vacance_2ans", "part_longue_parmi_vacants"]].describe().round(1)

,taux_vacance_lovac,taux_vacance_2ans,part_longue_parmi_vacants
count,90506.0,62619.0,93226.0
mean,10.4,4.7,43.7
std,4.9,3.4,16.1
min,0.0,0.0,0.0
25%,7.0,2.3,32.3
50%,9.6,3.9,44.4
75%,13.0,6.3,55.4
max,98.4,48.4,100.0


### Compléter les taux cachés par le taux de l'intercommunalité 

Le parc privé est caché pour environ la moitié des communes, presque toutes petites et rurales. Les supprimer reviendrait à retirer une grande partie du monde rural.

Pour ces communes, on utilise le taux de leur **intercommunalité (EPCI)**, calculé à partir des sommes des communes connues (vacants ÷ parc privé), pour le même millésime. Deux colonnes :
- `taux_impute` : vrai quand le taux vient de l'EPCI ;
- `taux_vacance_complete` : le taux de la commune s'il est connu, sinon celui de l'EPCI.

Limite : le taux de l'EPCI est surtout calculé sur ses plus grandes communes, dont le parc est connu.

In [51]:
# Taux de chaque EPCI, à partir des communes dont le taux est connu (sommes, pas moyenne de taux)
connues = lovac_long[lovac_long["taux_vacance_lovac"].notna()]
taux_epci = connues.groupby(["EPCI", "millesime"])[["vacants", "parc_prive"]].sum().reset_index()
taux_epci["taux_vacance_epci"] = ratio(taux_epci["vacants"], taux_epci["parc_prive"])

lovac_long = lovac_long.merge(taux_epci[["EPCI", "millesime", "taux_vacance_epci"]],
                              on=["EPCI", "millesime"], how="left")

# Taux de la commune s'il est connu, sinon celui de l'EPCI (signalé)
lovac_long["taux_impute"] = lovac_long["taux_vacance_lovac"].isna() & lovac_long["taux_vacance_epci"].notna()
lovac_long["taux_vacance_complete"] = lovac_long["taux_vacance_lovac"].fillna(lovac_long["taux_vacance_epci"])

print("Taux remplacés par celui de l'EPCI :", lovac_long["taux_impute"].sum())
print("Taux encore vides :", lovac_long["taux_vacance_complete"].isna().sum())

Taux remplacés par celui de l'EPCI : 118966
Taux encore vides : 34912


### Contrôler et enregistrer
Contrôles :
- une seule ligne par commune et par millésime ;
- la part de valeurs vides par millésime ;
- les taux compris entre 0 et 100 ;
- la cohérence : les vacants depuis plus de 2 ans font partie des vacants ;
- le taux national par millésime, calculé à partir des sommes, à comparer à la feuille « FR » du fichier (environ 11 % avant la rupture).

Le code commune est renommé `code_commune`, comme dans les autres tables.

In [52]:
print("Une ligne par commune et par millésime :",
      not lovac_long.duplicated(subset=["COM", "millesime"]).any())

# Part de valeurs vides par millésime (en %)
colonnes_a_verifier = ["vacants", "vacants_2ans", "parc_prive", "taux_vacance_lovac", "taux_vacance_complete"]
print((lovac_long[colonnes_a_verifier].isna().groupby(lovac_long["millesime"]).mean() * 100).round(1))

# Les taux doivent être entre 0 et 100
for col in ["taux_vacance_lovac", "taux_vacance_2ans", "part_longue_parmi_vacants", "taux_vacance_complete"]:
    hors_bornes = ((lovac_long[col] < 0) | (lovac_long[col] > 100)).sum()
    print(col, ": valeurs hors de [0, 100] =", hors_bornes)

# Cohérence : les vacants de plus de 2 ans sont une partie des vacants
print("Vacants 2 ans > vacants :", (lovac_long["vacants_2ans"] > lovac_long["vacants"]).sum())

# Taux national par millésime (sommes, pas moyenne des taux), sur les communes au taux connu
national = connues.groupby("millesime")[["vacants", "parc_prive"]].sum()
national["taux_%"] = (national["vacants"] / national["parc_prive"] * 100).round(1)
print(national)

# Enregistrement
lovac_long = lovac_long.rename(columns={"COM": "code_commune"})
lovac_long.to_parquet(DATA_OUT / "lovac_commune_annee.parquet", index=False)
print("Enregistré :", lovac_long.shape)

Une ligne par commune et par millésime : True
           vacants  vacants_2ans  parc_prive  taux_vacance_lovac  \
millesime                                                          
2020          21.2          58.1        53.3                56.9   
2021          20.6          57.4        53.1                56.7   
2022          21.4          58.3        53.0                56.8   
2023          20.6          56.8        52.9                56.6   
2024          19.7          77.1        52.7                56.3   
2025          26.9          63.4        52.5                57.6   
2026          19.7          50.9       100.0               100.0   

           taux_vacance_complete  
millesime                         
2020                         0.0  
2021                         0.0  
2022                         0.0  
2023                         0.0  
2024                         0.0  
2025                         0.0  
2026                       100.0  
taux_vacance_lovac : valeu

Enregistré : (244384, 18)


### Résultat du bloc 9

- `data/processed/lovac_commune_annee.parquet` : **244 384 lignes** (34 912 communes × 7 millésimes, 2020-2026), 18 colonnes.
- Effectifs : `vacants`, `vacants_2ans`, `parc_prive` (vides en cas de secret statistique ; `parc_prive` toujours vide en 2026).
- Dates : `millesime`, `annee_situation` (N-1), `annee_vente_associee` (N).
- Indicateurs en % : `taux_vacance_lovac`, `taux_vacance_2ans`, `part_longue_parmi_vacants`, et `taux_vacance_complete` (complété par le taux de l'EPCI, signalé par `taux_impute`).
- Signalements : `apres_rupture`, `alsace_moselle`, `mayotte`.
- Paris, Lyon et Marseille par arrondissement.

# Bloc 10 — Équipements (BPE) par commune

**Contributeurs :** Christian BAYILI · **Date :** 20/10/2026

Source : table construite par CB à partir de l'ancienne table cible (commune × année) et des Bases permanentes des équipements 2015 et 2020 de l'Insee. Les fichiers BPE bruts et le code de construction ne sont pas disponibles : cette table sert de source.

### Ce que contient la table

- 152 indicateurs `bpe_presence_...` : 1 = équipement présent, 0 = absent, vide = inconnu ;
- la BPE 2015 est associée aux ventes de 2016 à 2020, la BPE 2020 aux ventes de 2021 à 2025, et rien aux ventes de 2014 et 2015 ;
- seulement les communes qui ont eu des ventes.

### Ce qu'on en fait

- **On retire les prix et les nombres de ventes** : ils viennent de l'ancienne table cible, remplacée par le bloc 4.
- **Une ligne par commune × millésime BPE**, au lieu d'une par commune × année : les indicateurs sont les mêmes pour toutes les années qui utilisent le même millésime.
- **Le code commune en géographie 2026, arrondissements de Paris, Lyon et Marseille séparés**, comme dans les tables cibles du bloc 4.
- **Les valeurs inconnues sont gardées telles quelles.**

In [53]:
# On ne lit que les colonnes utiles : les codes, le millésime BPE et les 152 indicateurs
colonnes_utiles = ["code_commune", "code_commune_2026", "bpe_annee_source"]
equip = pd.read_csv(DATA_RAW / "cible_commune_annee_type_BPE_152_indicateurs.csv",
                    usecols=lambda c: c in colonnes_utiles or c.startswith("bpe_presence_"),
                    dtype={"code_commune": str, "code_commune_2026": str})
print("Chargé :", equip.shape)

colonnes_presence = [c for c in equip.columns if c.startswith("bpe_presence_")]
print("Indicateurs d'équipement :", len(colonnes_presence))

# Millésime BPE utilisé : vide pour les ventes de 2014 et 2015
print(equip["bpe_annee_source"].value_counts(dropna=False).sort_index())

Chargé : (367779, 155)
Indicateurs d'équipement : 152
bpe_annee_source
2015.0    153357
2020.0    153013
NaN        61409
Name: count, dtype: int64


### Une ligne par commune × millésime BPE

1. On retire les lignes sans millésime BPE (ventes de 2014 et 2015).
2. On construit le code commune en géographie 2026 : `code_commune_2026` de la table, sauf pour les arrondissements de Paris, Lyon et Marseille, qui gardent leur propre code.
3. On vérifie que les indicateurs sont bien identiques pour toutes les lignes d'une même commune et d'un même millésime : années différentes, ou anciennes communes fusionnées. `nunique(dropna=False)` compte les valeurs différentes, en comptant aussi le vide comme une valeur.
4. On ne garde qu'une ligne par commune × millésime.

In [54]:
# 1. Lignes avec un millésime BPE
equip = equip.dropna(subset=["bpe_annee_source"]).copy()
equip["millesime_bpe"] = equip["bpe_annee_source"].astype(int)

# 2. Code commune en géographie 2026, arrondissements séparés
arrondissement = equip["code_commune"].str.match(r"^(751\d\d|6938\d|132\d\d)$")
equip["code_commune_geo2026"] = equip["code_commune_2026"].where(~arrondissement, equip["code_commune"])

# 3. Contrôle : une seule valeur par indicateur pour chaque commune × millésime
cles = ["code_commune_geo2026", "millesime_bpe"]
ecarts = equip.groupby(cles)[colonnes_presence].nunique(dropna=False).max(axis=1)
print("Communes × millésime où les indicateurs diffèrent :", (ecarts > 1).sum())

# 4. Une ligne par commune × millésime
equip = equip.drop_duplicates(cles)[cles + colonnes_presence]
equip = equip.rename(columns={"code_commune_geo2026": "code_commune"})

print("Une commune × millésime = une ligne :", not equip.duplicated(["code_commune", "millesime_bpe"]).any())
print(equip.groupby("millesime_bpe").size())

Communes × millésime où les indicateurs diffèrent : 0
Une commune × millésime = une ligne : True
millesime_bpe
2015    32955
2020    33021
dtype: int64


### Enregistrer

On affiche d'abord la répartition des valeurs sur l'ensemble des indicateurs (présent, absent, inconnu), puis on enregistre.

In [55]:
valeurs = equip[colonnes_presence]
print("Présent :", round((valeurs == 1).mean().mean() * 100, 1), "%",
      "| absent :", round((valeurs == 0).mean().mean() * 100, 1), "%",
      "| inconnu :", round(valeurs.isna().mean().mean() * 100, 1), "%")

equip.to_parquet(DATA_OUT / "equipements_commune_millesime.parquet", index=False)
print("Enregistré :", equip.shape)

Présent : 10.3 % | absent : 2.5 % | inconnu : 87.2 %
Enregistré : (65976, 154)


### Résultat du bloc 10

- `data/processed/equipements_commune_millesime.parquet` : **65 976 lignes** (32 955 communes pour la BPE 2015, 33 021 pour la BPE 2020), 154 colonnes : `code_commune`, `millesime_bpe` et les 152 indicateurs `bpe_presence_...`.
- Code commune en géographie 2026, Paris, Lyon et Marseille par arrondissement : même clé que les tables cibles communales du bloc 4.
- Valeurs : 1 = présent (10,3 %), 0 = absent (2,5 %), vide = inconnu (87,2 %).
- Millésime à associer aux ventes : BPE 2015 pour 2016-2020, BPE 2020 pour 2021-2025, aucun pour 2014-2015.

# Bloc 11 — Taux des crédits immobiliers

**Contributeurs :** Gabrielle RICHERT · **Date :** 13/10/2026

Source : Banque de France (Webstat), taux annuel moyen des crédits nouveaux à l'habitat des particuliers, en %. Une valeur par mois, pour la France entière, de décembre 1998 à août 2026, observée en fin de mois.

### Ce qu'on prépare

- **Lecture** : les 5 premières lignes du fichier sont des métadonnées, pas des valeurs.
- **Conversion** : le taux est écrit avec une virgule (`3,21`), donc lu comme du texte ; on le convertit en nombre. La date est convertie en vraie date.
- **Trois tables**, une par pas de temps, comme les tables cibles du bloc 4 : le taux du mois, la moyenne du trimestre et la moyenne de l'année. Contrairement à une médiane de prix, un taux peut se moyenner : c'est un niveau, pas une distribution de ventes.
- **Toute la période est gardée**, y compris avant 2014 : elle pourra servir si l'on utilise le taux des mois précédant la vente.

In [56]:
chemin_taux = DATA_RAW / "Taux d'intérêt annuel des crédits immobiliers.csv"

# 6 lignes de métadonnées en tête de fichier, pas de ligne d'en-tête
taux = pd.read_csv(chemin_taux, sep=";", skiprows=6, header=None, names=["date", "taux"])

print(taux.shape)
print(taux.dtypes)
taux.head(3)

(333, 2)
date    str
taux    str
dtype: object


,date,taux
0,2026-08-31,"3,21"
1,2026-07-31,"3,18"
2,2026-06-30,"3,17"


### Convertir les dates et les taux, ajouter les périodes

- **Taux** : la virgule décimale est remplacée par un point, puis le texte est converti en nombre (`3,21` devient `3.21`).
- **Date** : le texte `2026-08-31` devient une vraie date.
- **Ordre chronologique** : le fichier commence par le mois le plus récent ; on le remet dans l'ordre du temps.
- **Périodes** : on ajoute l'année, le mois et le trimestre, les mêmes clés que dans les tables cibles du bloc 4.

In [57]:
# Texte -> nombre décimal : la virgule est remplacée par un point
taux["taux"] = pd.to_numeric(taux["taux"].str.replace(",", ".", regex=False))

# Texte -> date
taux["date"] = pd.to_datetime(taux["date"])

# Du plus ancien au plus récent
taux = taux.sort_values("date").reset_index(drop=True)

# Mêmes clés de période que les tables cibles
taux["annee"] = taux["date"].dt.year
taux["mois"] = taux["date"].dt.month
taux["trimestre"] = (taux["mois"] - 1) // 3 + 1

print(taux.dtypes)
print("Du", taux["date"].min().date(), "au", taux["date"].max().date(), "|", len(taux), "mois")
print("Mois en double :", taux.duplicated(["annee", "mois"]).sum())
taux["taux"].describe().round(2)

date         datetime64[us]
taux                float64
annee                 int32
mois                  int32
trimestre             int32
dtype: object
Du 1998-12-31 au 2026-08-31 | 333 mois
Mois en double : 0


count    333.00
mean       3.37
std        1.40
min        1.10
25%        2.18
50%        3.51
75%        4.23
max        6.10
Name: taux, dtype: float64

### Calculer les trois tables et enregistrer

- **Mois** : le taux tel quel.
- **Trimestre** et **année** : la moyenne des taux mensuels de la période.

Les clés sont les mêmes que dans les tables cibles du bloc 4 : `annee` et `mois`, `annee` et `trimestre`, ou `annee` seule.

In [58]:
taux_mois = taux[["annee", "mois", "taux"]]
taux_trimestre = taux.groupby(["annee", "trimestre"], as_index=False)["taux"].mean().round(2)
taux_annee = taux.groupby("annee", as_index=False)["taux"].mean().round(2)

taux_mois.to_parquet(DATA_OUT / "taux_credit_mois.parquet", index=False)
taux_trimestre.to_parquet(DATA_OUT / "taux_credit_trimestre.parquet", index=False)
taux_annee.to_parquet(DATA_OUT / "taux_credit_annee.parquet", index=False)

print("Mois :", len(taux_mois), "| trimestres :", len(taux_trimestre), "| années :", len(taux_annee))
taux_annee[taux_annee["annee"] >= 2014]

Mois : 333 | trimestres : 112 | années : 29


,annee,taux
16,2014,2.96
17,2015,2.32
18,2016,1.88
19,2017,1.60
20,2018,1.55
21,2019,1.36
22,2020,1.26
23,2021,1.15
24,2022,1.47
25,2023,3.02


### Résultat du bloc 11

Trois tables dans `data/processed/` :

| Fichier | Une ligne = | Lignes |
|---|---|---|
| `taux_credit_mois.parquet` | un mois | 333 |
| `taux_credit_trimestre.parquet` | un trimestre (moyenne des mois) | 112 |
| `taux_credit_annee.parquet` | une année (moyenne des mois) | 29 |

- Période : décembre 1998 à août 2026. Les années 1998 (un seul mois) et 2026 (huit mois) sont incomplètes : leur moyenne ne porte que sur les mois disponibles.
- Avant 2003, la valeur change une fois par trimestre et se répète sur les trois mois.
- Taux en %, France entière.

# Bloc 12 — Équipements par commune (BPE 2025)

**Contributeurs :** Cécile NAUDET · **Date :** 28/10/2026

Source : Insee, base permanente des équipements (BPE) 2025, une ligne par équipement (audit dans l'EDA complémentaire).

### Ce qu'on prépare

- **Six domaines** : services, commerces, santé, transports, sports, tourisme. L'enseignement est exclu (équipements de l'Éducation nationale absents du fichier).
- **Seuls les types présents dans la BPE en évolution sont comptés** : les nombres sont ainsi comparables à ceux de l'EDA complémentaire (97,4 % des bassins à moins de 1 % d'écart).
- **Une ligne = une commune** (géographie 2026, arrondissements de Paris, Lyon et Marseille séparés), avec le nombre d'équipements de chaque domaine. La table s'additionne à n'importe quelle maille ; une commune absente n'a aucun équipement recensé.

### Lire la BPE 2025 et la liste des types de la BPE en évolution

In [ ]:
DOMAINES = {"A": "services", "B": "commerces", "D": "sante",
            "E": "transports", "F": "sports", "G": "tourisme"}   # C (enseignement) exclu

# --- Lecture : 4 colonnes utiles sur 95 ---
bpe = pd.read_parquet(DATA_RAW / "BPE25.parquet", columns=["DEPCOM", "DEP", "DOM", "TYPEQU"])

# --- Types de la BPE en évolution (2025, bassins de vie de métropole) ---
evo = pd.read_csv(DATA_RAW / "DS_BPE_EVOLUTION_2025_data.csv", sep=None, engine="python", dtype=str,
                  usecols=["GEO_OBJECT", "GEO", "TIME_PERIOD", "FACILITY_TYPE"])
types_evo = set(evo.loc[(evo["GEO_OBJECT"] == "BV2022") & (evo["TIME_PERIOD"] == "2025")
                        & ~evo["GEO"].str.startswith("97"), "FACILITY_TYPE"])
print("Équipements :", len(bpe), "| types de la BPE en évolution :", len(types_evo))

### Garder la métropole, les 6 domaines et les types communs, puis rattacher à la commune 2026

Le code commune de chaque équipement est rattaché à son code 2026 par la table de liaison du bloc 1. Les arrondissements gardent leur propre code.

In [ ]:
# --- Métropole, 6 domaines, types communs aux deux fichiers ---
n_brut = len(bpe)
bpe = bpe[~bpe["DEP"].str.startswith("97") & bpe["DOM"].isin(DOMAINES) & bpe["TYPEQU"].isin(types_evo)].copy()

# --- Rattachement à la commune 2026 (table de liaison du bloc 1), arrondissements gardés ---
liaison = pd.read_parquet(DATA_OUT / "liaison_commune_bv2022.parquet", columns=["CODGEO_INI", "CODGEO_2026"])
bpe = bpe.merge(liaison, left_on="DEPCOM", right_on="CODGEO_INI", how="left", validate="many_to_one")
arrondissement = bpe["DEPCOM"].str.match(r"^(751\d\d|6938\d|132\d\d)$")
bpe["code_commune"] = bpe["CODGEO_2026"].where(~arrondissement, bpe["DEPCOM"])

### Compter les équipements par commune et par domaine, contrôler et enregistrer

In [ ]:
# --- Nombre d'équipements par commune et par domaine ---
bpe["domaine"] = bpe["DOM"].map(DOMAINES)
equip25 = (bpe.groupby(["code_commune", "domaine"]).size()
              .unstack(fill_value=0)
              .add_prefix("nb_equip_")
              .reset_index())
equip25.columns.name = None
equip25["millesime_bpe"] = 2025

# --- Contrôles et enregistrement ---
colonnes_nb = [c for c in equip25.columns if c.startswith("nb_equip_")]
print("Bloc 12 - équipements gardés :", len(bpe), "sur", n_brut,
      "| sans commune 2026 :", bpe["code_commune"].isna().sum(),
      "| communes :", len(equip25),
      "| une commune = une ligne :", equip25["code_commune"].is_unique,
      "| aucun équipement perdu :", equip25[colonnes_nb].sum().sum() == bpe["code_commune"].notna().sum())
equip25.to_parquet(DATA_OUT / "equipements_commune_2025.parquet", index=False)

### Résultat du bloc 12

- `data/processed/equipements_commune_2025.parquet` : **34 786 communes et arrondissements**, une colonne `nb_equip_` par domaine (6) et `millesime_bpe` = 2025.
- 2 366 270 équipements gardés sur 2 921 770 (métropole, 6 domaines, types communs avec la BPE en évolution) ; tous rattachés à une commune 2026, aucun perdu.
- Une commune absente de la table n'a aucun équipement recensé : son nombre vaut 0 à l'agrégation.